# <center> Final project

### <center> Resume recommender for Job position
<center> <img src=https://media.licdn.com/dms/image/C5612AQEg5ghSochF8Q/article-cover_image-shrink_720_1280/0/1632544677267?e=2147483647&v=beta&t=1_Hpp19AXksV23OSE3z1Tpw35J6_P17WvA4pvNTmYHA align="center" width="300"/> </center>

**Business Task**:
To create a recommender system that recommends the best-matched resumes to vacancy.

*Description of business situation*: it is a big company. There are a lot of open vacancies (corresponds to different job positions) and pull of resumes (HR specialist has already collected them). Now HR specialist needs to find the best candidate for vacancy from these pull of resumes.

**Technical Task**
* Using Natural Language Processing (NLP) techniques to process resumes and job listings and sort resumes by similarity score in descending order, to create this ultimate resume screening tool.

**Type of ML task**
* Recommender System based on similarity score
* Dimensionality reduction by SVD/ TSNE (for visualization) + Clusterization (Kmeans, DBSCAN, Agglomerative clustering) + Recommender System based on similarity score
* Topic modeling (NMF) + Recommender System based on similarity score

**Metrics:**
* Recommender System: cosine similarity 
* Clusterization: Silhouette coefficient, Gap statistic - to get optimal number of clusters
* Topic modeling (NMF): coherence score - to evaluate the best number of topics

**Data:** The dataset was found on Kaggle, with a total of 8,653 entries of applicant experiences and ~80K job listings (before dataset cleaning stage)

# <p style="text-align:center;font-size:100%;">0. Install and Import</p>

In [ ]:
# Install libs
!pip install kneed
!pip install wordcloud

# Import libs
import os
import pandas as pd
import numpy as np
import re # regular expression
import pickle # for save model/ data

from collections import Counter, defaultdict
from operator import itemgetter
from time import time

# Libs for visualization
import matplotlib.pyplot as plt
import seaborn as sns
import folium # map visualization
import wordcloud
from kneed import KneeLocator # Knee-point detection

# Text preprocessing
import nltk
import subprocess

# Download and unzip wordnet
try:
    nltk.data.find("wordnet.zip")
except:
    nltk.download("wordnet", download_dir="/kaggle/working/")
    command = "unzip /kaggle/working/corpora/wordnet.zip -d /kaggle/working/corpora"
    subprocess.run(command.split())
    nltk.data.path.append("/kaggle/working/")
    
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords #list of lexical stop words
from nltk import word_tokenize, pos_tag, pos_tag_sents
nltk.download("stopwords")
from sklearn.feature_extraction.text import TfidfVectorizer
from string import punctuation #sets of punctuation

# Metrics
from scipy.stats import normaltest # D'Agostino's K-squared test
from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics import silhouette_score, pairwise_distances
from gensim.models.coherencemodel import CoherenceModel

# Dimensionality reduction
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE

# Clustering algorithms, Topic modeling
from sklearn.cluster import KMeans, MiniBatchKMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import NMF
from gensim.models.nmf import Nmf
from gensim import corpora
from gensim.corpora.dictionary import Dictionary

import warnings
warnings.filterwarnings("ignore")

# Input data files are available in the read-only "../input/" directory
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Fixing RANDOM_SEED to make experiment repetable
RANDOM_SEED = 42

# Fixing package versions to make experiment repetable
!pip freeze > requirements.txt

In [ ]:
# Load datasets
DATA_DIR = "/kaggle/input/job-recommendation-datasets/" #"data/"

job_description = pd.read_csv(DATA_DIR+"Combined_Jobs_Final.csv")
position_of_interest = pd.read_csv(DATA_DIR +"Positions_Of_Interest.csv") # position of interest
job_data = pd.read_csv(DATA_DIR+"job_data.csv") # job data
resumes = pd.read_csv(DATA_DIR+"Experience.csv") # experience of candidate
job_views = pd.read_csv(DATA_DIR+"Job_Views.csv") # job views

#### Let's look at the content of each datasets

In [ ]:
# Vacancies information (full version)
display(job_description.head(2))
print(f"\nNumber of rows and columns in dataset: {job_description.shape}")

In [ ]:
# Applicant's position of interest
display(position_of_interest.head(2))
print(f"\nNumber of rows and columns in dataset: {position_of_interest.shape}")

In [ ]:
# Short version of vacancy dataset: Job_ID and merged information from Job_title and Job_description
# it's look like cleaned vacancy dataset
display(job_data.head(2))
print(f"\nNumber of rows and columns in dataset: {job_data.shape}")

In [ ]:
# Resumes dataset
display(resumes.head(5))
print(f"\nNumber of rows and columns in dataset: {resumes.shape}")

In [ ]:
# Information about Job views by Applicant
display(job_views.head(5))
print(f"\nNumber of rows and columns in dataset: {job_views.shape}")

We have 5 datasets:
- Vacancies information (full version)
- Applicant's position of interest
- Short version of vacancy dataset: Job_ID and merged information from Job_title and Job_description
- Resumes dataset
- Information about Job views by Applicant

In this project I **use only 2 datasets**: *Vacancies* and *Resumes*.

- Dataset with job data looks like processed full version of vacancy dataset (it is not so interisting to get it for investigation).
- *Position of interest* is about prefereble job position for applicant. And as you can see sometimes applicant's job experience is different than position of interest. For example:
    - Applicant 10003: job experience as "*maintenance technician*" <-> position of interest "*security officer*"
- *Job views* describes information about job views by Applicant. Maybe this dataset will be useful for further investigation, but not in this case.

***

# <p style="text-align:center;font-size:100%;">1. Checking dataset: missing values, duplicates, outliers</p>

## <center> VACANCIES DATASET

### `Total information`

Let's read the dataset into the DataFrame *job_description* and will have a look at the *shape, columns, column data types and the first 5 rows of the data*.

This will give a brief overview of the data at hand.

In [ ]:
# Reading csv-file
print(f"Data shape: {job_description.shape}\n")

display(job_description.head(5))
display(job_description.info())

### **Exploring the content of variables**

The dataset is dedicated to the vacancies and contains 84,090 rows and 23 columns.

Let's divide all features into 4 groups:
- job description
- candidate requirements
- location
- non-informative features

*Job description*:
- Job.ID: nominal, from 1-digit to 6-digit integral number uniquely assigned to each job posting's.
- Title: this column contains of information about job position and company name by "@" symbol ("Server @ Tacolicious")
- Position: job position
- Company: employer name
- Job.Description: description of job position
- Salary: numeric, salary per hour (/day)
- Employment.Type: type of employment
- Industry: industry of the company/ job
- Slug: this column contains of information about city name, state code, company and job position by "-" symbol ("palo-alto-ca-tacolicious-server")

*Candidate requirements*:
- Requirements: job requirements (for candidate). This columns is empty -> move it to *Non-informative features*
- Education.Required: required education level

*Location*:
- City: full name of city
- State.Name: full name of state
- State.Code: state abbreviations
- Address: address of the company
- Latitude: latitude of the company location
- Longitude: longitude of the company location

*Non-informative features*:
- Provider: nominal, 1-digit integral number. There are three variants: 1,2,3.
- Status: all rows have the same "status" value - *open*.
- Listing.Start: date objects
- Listing.End: date objects
- Created.At: date objects
- Updated.At: date objects

Some observations about the data:
* There are columns with duplicated information:
    - State.Name and State.Code
    - Slug
     
The most interesting groups for our task is *Job description* and *Candidate requirements*. Unfortunately, in further investigation I'll see that the last group's columns are empty or describes a small part of data.

In [ ]:
"""
Define Unique categories in each column
"""
unique_list = []

for col in job_description.columns:
    # creating tuple: column name, number of unique values, type
    item = (col, job_description[col].nunique(), job_description[col].dtypes, job_description[col].unique())
    unique_list.append(item)
    
unique_counts = pd.DataFrame(
    unique_list,
    columns=["Column_Name", "Num_Unique", "Type", "Unique_category"]
).sort_values(by="Num_Unique")


display(unique_counts)

***

### Candidate requirements features

- Requirements
- Education.Required

*Requirements has only empty values. We should drop this column*

***

In [ ]:
"""
Let's check unique categories in Education.Required column in details
"""
edu_level = job_description["Education.Required"].value_counts().reset_index()
edu_level.columns = ["Education.Required", "count"]

#define Seaborn color palette to use
colors = sns.color_palette("pastel")

#create pie chart
plt.figure(facecolor='white', figsize=(6,8))
plt.pie(edu_level["count"], colors = colors, autopct="%.0f%%", pctdistance=1.1)
plt.title("Groups of required education level", fontsize=16)
plt.legend(edu_level["Education.Required"], loc="upper center", bbox_to_anchor=(0.5, -0.04), ncol=3, fontsize=12)

plt.show()

Some observations:
* The most part of data has not specified information about education level - 75%

It means that this column will be useless during further investigation.

***

### Job description features
- Job.ID - each row describes each unique vacancy
- Position
- Company
- Job.Description
- Salary
- Employment.Type
- Industry

Title and Slug columns duplicate information from Position, Company, Location features. I guess that I won't use it in this study

***

In [ ]:
"""
Let's check unique categories in Industry column in details
"""
industry = job_description["Industry"].value_counts().reset_index()
industry.columns = ["Industry", "count"]

#define Seaborn color palette to use
colors = sns.color_palette("pastel")

#create pie chart
plt.figure(facecolor='white', figsize=(6,8))
plt.pie(industry["count"], colors=colors, autopct="%.0f%%", pctdistance=1.1)
plt.title("Industries of employer companies", fontsize=16)
plt.legend(industry["Industry"], loc="upper center", bbox_to_anchor=(0.5, -0.04), ncol=3, fontsize=12)

plt.show()

Some observations:
* Based at the pie chart, there are 5 unique industries:
    - Food and Beverages ~ 60%
    - Transportation - 15%
    - Retail - 8%
    - Office Administration - 4%
    - Care giving ~ less than 1%
* This column has only 267 not null data. It means that this feature is also useless for this study

In [ ]:
"""
Let's check unique categories in Employment.Type column in details
"""
round(job_description["Employment.Type"].value_counts(normalize=True), 2)

As you can see **Seasonal/Temp** type of employment has the second version - *Temporary/seasonal* (with less examples number).

We should to merge these examples into one type: *Seasonal/Temp*

In [ ]:
# replacing "Temporary/seasonal" value to "Seasonal/Temp" in type of employment column
job_description.loc[:, "Employment.Type"] = job_description["Employment.Type"].replace(["Temporary/seasonal"], "Seasonal/Temp")

# getting temp df with counts of Employment.Type
employment_type = job_description["Employment.Type"].value_counts().reset_index()
employment_type.columns = ["Employment.Type", "count"]

# getting popular types of employment (taggig unpopular as "other")
popular_emp_type = employment_type[employment_type["count"] >= 1000]["Employment.Type"].tolist()
employment_type["Employment.Type"] = employment_type["Employment.Type"].apply(lambda x: x if x in popular_emp_type else "Other")

# aggregate the same values into one
employment_type = employment_type.groupby(["Employment.Type"])["count"].sum().reset_index()

#define Seaborn color palette to use
colors = sns.color_palette("pastel")

#create pie chart
plt.figure(facecolor='white', figsize=(6,8))
plt.pie(employment_type["count"], colors = colors, autopct="%.0f%%", pctdistance=1.1)
plt.title("Type of employment", fontsize=16)
plt.legend(employment_type["Employment.Type"], loc="upper center", bbox_to_anchor=(0.5, -0.04), ncol=3, fontsize=12)

plt.show()

Some observations:
* The most popular type of employment is Part time - 40%
* The least popular type of employment belongs to Other group which includes Intern, Full-time and Contract - 1%

I'm not sure that this feature is meaningful for our task, I'll drop it later

In [ ]:
"""
Let's check the distribution of Salary
"""
# coordinate systems visualization
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15,4))

# create histogram
sns.histplot(
    job_description["Salary"],
    bins=25,
    kde=True,
    ax=axes[0]
);
axes[0].set_title("Salary distribution", fontsize=16);
axes[0].set_xlabel("Salary ($ per hour/ day)")

# create boxplot
sns.boxplot(
    job_description["Salary"],
    orient="h",
    width=0.9,
    ax=axes[1]
);
axes[1].set_title("Salary distribution");
axes[1].set_xlabel("Salary ($ per hour)");
axes[1].grid()

job_description["Salary"].describe()

Some observations:
* based on the histogram and boxplot of Salary distribution I can conclude that the data aren't normally distributed. There are outliers.
* anyway, this column has only 229 not null rows, that's why it has no sense to study it

In [ ]:
"""
Let's check unique categories in Position column in details
"""
job_description["Position"].value_counts()

In [ ]:
job_title = job_description["Position"].value_counts().reset_index()[:20]
job_title.columns = ["Position", "count"]

sns.barplot(x="count", y="Position", data=job_title, color="blue", palette="hls").set(title="Top-20 job title from vacancies");

Top-5 job position in Vacancies:
* Administrative assistant
* Customer care representative
* Accounts payable clerk
* Accounting clerk
* Sales representative

Based on this small part of job position I can conclude that there are cases when one job position has different variation:
* *Customer service representative* <-> *Seasonal CSR*
In this case the word *seasonal* should be remove to Employment.Type column.

So, I guess that it is popular problem when one job position has different variation of title name. I can check it manually in order to get "clean" job titles, but it needs time. Maybe it will be good for further study.

***

### Location features
- City
- State.Name
- State.Code
- Address
- Latitude, Longitude

***

In [ ]:
"""
Let's check unique categories in State.Name and State.Code columns in details
"""
state_name_code = job_description.groupby("State.Code")["State.Name"].value_counts().reset_index(name="count").sort_values(["count"], ascending=False).reset_index()
state_name_code.drop("index", axis=1)

Some observations:
* 51 USA states
* The most vacancies are from California companies
* The least vacancies are from Wyoming companies

In [ ]:
"""
Let's check unique categories in City column in details
"""
city_by_state = job_description.groupby("State.Code")["City"].value_counts().reset_index(name="count")

print(f"\nNumber of cities from different states: {city_by_state.shape[0]}\n")
display(city_by_state[:10])

In [ ]:
# Checking for duplicate city names from different states
cities = city_by_state.groupby(["City"])["State.Code"].nunique().reset_index()
townships = cities[cities["State.Code"] != 1]

print(f"\nNumber of cities-townships: {townships.shape[0]}\n")
display(townships[:10])

Some observations:
- This dataset consists of information about vacancies from 7203 cities
- There are 870 cities-townships. It is a case when there is the same name of city in different states. This is an important point for the stage of filling Missing values.

In [ ]:
"""
Let's check unique categories in Company column in details
"""
company_by_state = job_description.groupby("State.Code")["Company"].value_counts().reset_index(name="count")

print(f"\nNumber of companies from different states: {company_by_state.shape[0]}\n")
display(company_by_state[:10])

In [ ]:
# Checking for duplicate company names from different states
companies = company_by_state.groupby(["Company"])["State.Code"].nunique().reset_index()
duplicated_name_of_company = companies[companies["State.Code"] != 1]

print(f"\nNumber of companies with the same name from different states: {duplicated_name_of_company.shape[0]}\n")
display(duplicated_name_of_company[:20])

In [ ]:
"""
Let's visualize companies location in a map context
"""
# Create city_map dataset for visualisation
city_cols = list(["City"] + ["Latitude"] + ["Longitude"])
city_data = job_description[city_cols]
city_data.dropna(inplace=True) # delete rows with missing values


# latitude and longitude of USA
latitude = 33.247875
longitude = -83.441162

city_map = folium.Map(location=[latitude, longitude], zoom_start=11)

# add markers to map
for lat, lng, label in zip(city_data["Latitude"], city_data["Longitude"], city_data["City"]):
    label = folium.Popup(label, parse_html=True)
    folium.CircleMarker(
        [lat, lng],
        radius=5,
        popup=label,
        color="blue",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.7,
        parse_html=False).add_to(city_map)  
    
city_map.save("./city_map.html")
# city_map

***Some observations about the data:***

* Number of required education level: 4 specific levels and 1 not specified (about 75% of data).

* This dataset presents of vacancies from 5 different industries as:
    - Food and Beverages
    - Transportation
    - Retail
    - Office Administration
    - Care Giving and Other
    
    
* Number of unique employment type: 7.
    - Part-Time
    - Full-Time/Part-Time
    - Per Diem
    - Intern
    - Full-Time
    - Contract
    - "Seasonal/Temp" and "Temporary/seasonal" are the same. We should check it further


* Top-5 job position in Vacancies: Administrative assistant, Customer care representative, Accounts payable clerk, Accounting clerk, Sales representative. It belongs to Office Administration and Retail industries. There is a problem when one job position has different variation of title name. We can check it manually and replace to "true value" by using dictionary and map() method.

* Number of states: 51. The most vacancies from companies based in California.

* Number of unique city names: 5503. **But** after grouping city with state code I see more than 7k rows of cities. It means that there are cities with similar names in different states (usually this city called as township). There are 870 cities-townships.

* This dataset consists of vacancies from 8335 companies. **But** after grouping company names with state code I saw the same situation as with cities-townships. In real there are 15566 different companies. Number of companies with the same name from different states: 1502

* Number of unique job position: 36245. I assume that there are fewer unique job positions, because there are examples where the same job position is written differently (caps lock, added company name/ employment type). We can check it further
    - "Sales Representative / Sales Associate ( Entry Level )" / "Sales Associate" / "SALES ASSOCIATE (Herndon, VA - PT)" / "Sales Associate - Slots A Fun Shop (PT)"

***

### Checking missing values and full duplicates

In [ ]:
"""
Let`s check our data for Missing values (MV)
"""
print(f"Total number of rows, columns: {job_description.shape}")

# duplicates
all_columns = list(job_description.columns)
mask_dupl = job_description.duplicated(subset=all_columns) 
data_duplicates = job_description[mask_dupl]
print(f"\nNumber of found duplicates: {data_duplicates.shape[0]}")

# features with missing values
nulls_data = job_description.isnull().sum()
nulls = nulls_data[nulls_data > 0]
print(f"\nFeatures with nulls: {nulls_data[nulls_data>0].count()}\n{nulls}")

 From the above list we see that there are lot of NaN values, perform data cleansing for each and every column

#### **Subsetting only needed columns and not considering the columns that are not necessary:**

*Columns with more than 40% of missing values*:
- Industry
- Requirements
- Salary
- Address

*Columns with duplicated information*:
- State.Name
- Slug
- Title

*Non informative columns*:
- Provider
- Status
- Listing.Start
- Listing.End
- Created.At
- Updated.At'

*Columns for further work*:
* Job.ID
* Position
* Company
* City
* State.Code
* Latitude
* Longitude
* Job.Description
* Employment.Type
* Education.Required

In [ ]:
# get columns for further work
cols = list(["Job.ID"]+["Position"]+["Company"]+["City"]+["State.Code"]+\
    ["Latitude"]+["Longitude"]+["Employment.Type"]+["Job.Description"]+["Education.Required"])

jobs_temp = job_description[cols]
jobs_temp.columns = ["Job_ID", "Position", "Company", "City", "State_code",
                      "Latitude", "Longitude", "Employment_type", "Job_description", "Edu_required"]
jobs_temp.head()

In [ ]:
# checking for the null values again
jobs_temp.isnull().sum()

## `City` column

Let's select NaN rows of City and check is there any value in Latitude and Longitude columns. If it is true, then it is possible to recover NaN values in City column.

In [ ]:
"""
Selecting NaN rows of City
"""
nan_city = jobs_temp[jobs_temp["City"].isnull()]

print(nan_city.shape)
display(nan_city.head())

So, my hypothesis is false. In this case columns with information about Latitude and Longitude are only duplicate information from city columns. I can delete them.

***

In [ ]:
# Look at the companies list in cities with no name
nan_city.groupby(["Company"])["City"].count()

We see that there are only 9 unique companies cities that are having NaN values. So I manually adding their head quarters with the help of google search


In [ ]:
"""
Replacing NaN with thier headquarters location
I created 2 dictionary with City name and State code because there are cities with the same name in different states.
"""
jobs_temp.loc[:, "Company"] = jobs_temp.loc[:, "Company"].replace(["Genesis Health Systems"], "Genesis Health System")

# # we should check every cities of these companies "is this city township or unique?" For example:
# jobs_temp[jobs_temp["City"] == "Washington"]["State_code"].value_counts()

city_of_headquarters = {
    "Academic Year In America": "Stamford",
    "CBS Healthcare Services and Staffing": "Urbandale",
    "CHI Payment Systems": "Westlake Village",
    "Driveline Retail": "Coppell",
    "Educational Testing Services": "Princeton",
    "Genesis Health System": "Davenport",
    "Home Instead Senior Care": "Omaha",
    "St. Francis Hospital": "Hartford",
    "Volvo Group": "Washington"
}

state_code_of_headquarters = {
    "Academic Year In America": "CT",
    "CBS Healthcare Services and Staffing": "IA",
    "CHI Payment Systems": "CA",
    "Driveline Retail": "TX",
    "Educational Testing Services": "NJ",
    "Genesis Health System": "IA",
    "Home Instead Senior Care": "NE",
    "St. Francis Hospital": "CT",
    "Volvo Group": "DC"
}

# Getting a list with MV rows
city = list(jobs_temp[jobs_temp["City"].isna()].index)
state_code = list(jobs_temp[jobs_temp["State_code"].isna()].index)

# Filling MV by dict
jobs_temp.loc[city, "City"] = jobs_temp.loc[city, "Company"].map(city_of_headquarters)
jobs_temp.loc[state_code, "State_code"] = jobs_temp.loc[state_code, "Company"].map(state_code_of_headquarters)

display(jobs_temp.head())

In [ ]:
jobs_temp.isnull().sum()

## `Employment type`

In [ ]:
"""
Selecting NaN rows of Employment type
"""
nan_emp_type = jobs_temp[jobs_temp["Employment_type"].isnull()]

display(nan_emp_type)

There are only 10 rows with MV. All of them belongs to Uber company, which looks for driving partner. So I assume it look like part-time/full time job position.

In [ ]:
# replacing na values with part time/full time
jobs_temp.loc[:, "Employment_type"] = jobs_temp["Employment_type"].fillna("Full-Time/Part-Time")

jobs_temp.groupby(["Employment_type"])["Company"].count()

## `State code`

In [ ]:
"""
Selecting NaN rows of State code
"""
nan_state_code = jobs_temp[jobs_temp["State_code"].isnull()]

display(nan_state_code.head(10))

In [ ]:
# Look at the lat, lng coordinates of cities with no State code
nan_state_code.groupby(["City"])[["Latitude", "Longitude"]].mean()

All of these cities based on Puerto Rico. That's why they have not any state code. But they are part of USA.

*Puerto Rico officially the Commonwealth of Puerto Rico is a Caribbean island and unincorporated territory of the United States*

I want to add as state code **PR**

In [ ]:
# fill missing values in state code with PR tag (Puerto Rico)
jobs_temp.loc[:, "State_code"] = jobs_temp["State_code"].fillna("PR")

display(jobs_temp[jobs_temp["State_code"] == "PR"].head())

## `Required education level`

I want to check two points:
- what types of job positions have NaN value in Education_required column
- and is it possible to fill missing values by the other examples in dataset?

In [ ]:
"""
Selecting NaN rows of Education level
"""
nan_edu_level = jobs_temp[jobs_temp["Edu_required"].isnull()]

display(nan_edu_level.head())

In [ ]:
"""
Checking the type of job position with NaN value in Edu_requierd column
"""
nan_edu_level["Position"].value_counts()

In [ ]:
"""
Let's look at the required education level corresponds to "Server" and "Line Cook" positions (they have high level of missing values)
"""
print("\nRequired education level at the Server position\n")
display(jobs_temp[jobs_temp["Position"] == "Server"]["Edu_required"].value_counts())

print("\nRequired education level at the Line Cook position\n")
display(jobs_temp[jobs_temp["Position"] == "Line Cook"]["Edu_required"].value_counts())

As you can see, it is not so obvious how to fill missing values correctly. Because *Not specified* and *High school diploma* are about different.

Also, earlier I've pointed that the most popular tag for this column is *Not specified* (75%).

Maybe it has more sense to drop these rows or don't use this feature at all.

***

As you can see in the list of columns above, there is a lot of information about location of the job (city, State.Name, State.Code, Address, Latitude, Longitude), which we will not use to recommend resume.

The rest of the columns either don’t seem to be adding many values to our use case, or it has a lot of missing values.

So, I want to use only 3 columns from Vacancies dataset:
* Job_ID
* Position (or job title)
* Job description

In [ ]:
"""
Get final_jobs dataset with 3 columns
Drop rows with MV in "Job_description" column
"""
cols = list(["Job_ID"]+["Position"]+["Job_description"])
final_jobs = jobs_temp[cols]
final_jobs.columns = ["Job_ID", "Job_position", "Job_description"]
display(final_jobs.head(2))

final_jobs = final_jobs.dropna(subset=["Job_description"])
final_jobs.shape

In [ ]:
# Check MV one more time
final_jobs.isnull().sum()

#### Deleting full duplicates in Job_description columns from Vacancies dataset

In [ ]:
"""
Let`s check our Job_description column to duplicates
"""
duplicate = final_jobs[final_jobs.duplicated("Job_description")]

print("\nDuplicated rows in Job_description: \n")
display(duplicate.head(3))
print(f"\nNumber of duplicated rows: {duplicate.shape[0]}")

In [ ]:
"""
Delete duplicates by drop_duplicates() method
Create new df vacancies_dedupped - is cleaned from duplicates
"""
vacancies_dedupped = final_jobs.drop_duplicates(subset=["Job_description"])
print(f"Total number of rows after deleting duplicates: {vacancies_dedupped.shape[0]}")

***

## <center> RESUMES DATASET

Let's look at the resumes dataset in details.

In [ ]:
display(resumes.head())
print(f"\nShape of resumes dataset: {resumes.shape}\n")
display(resumes.info())

### **Exploring the content of variables**

The dataset is dedicated to the Resumes and contains 8,653 rows and 13 columns.

Let's divide all features into 4 groups:
- Applicant information
    * Applicant_ID
- Job experience: 
    * Job_title (previous work places), Employer.Name, Duration of a job (Start.Date, End.Date), Job_description, Salary, Can.Contact.Employer
- Employer location:
    * City, State.Name, State.Code,
- Non-informative features:
    * Created.At, Updated.At
     
The most interesting columns for our task are *Job title* and *Job descriprion*. Anyway, information about *Duration of a job* is also important in order to get total years of applicant's work experience. But the Requirenments column is empty in Vacancies dataset, in this case information about work experience years becomes meaningless.

In [ ]:
# I modify the column name so that I can use df dot column name more easily
resumes = resumes.rename(columns={"Applicant.ID": "Applicant_ID",
                                  "Position.Name": "Job_title",
                                  "Job.Description": "Job_description"})

print(f"\nNumber of unique applicant ID: {resumes.Applicant_ID.nunique()}")

In [ ]:
"""
Distribution of Missing Values according to the data
"""
cols_with_null = resumes.isnull().sum()

colors = ["blue", "yellow"]

fig = plt.figure(figsize=(10,4))
cols = cols_with_null.index
ax = sns.heatmap(
    resumes[cols].isnull(),
    cmap=sns.color_palette(colors)
)

The graph shows that the concentrations of Missing Values are on the next points:
* Location features,
* Information about previous job place
    - work period
    - salary
    - can.contact.employer
    - job description and job title

It will be enough to delete uninformative columns and rows with MV in Job description and Job title columns

In [ ]:
"""
For example, for applicant_id 10001
its job description is shown as Nan for the first three rows, hence these observations will be removed and won’t be considered in the dataset.
"""
resumes[resumes["Applicant_ID"] == 10001]

In [ ]:
# But, as we know, that there is implicit MV too - special words/ symbols. Let's look at the example - "none"
resumes.iloc[45]

In [ ]:
# None example
resumes[resumes["Applicant_ID"] == 1105]["Job_description"].iloc[0]

So, this implicit missing values not a simple as you can see.
* firtsly, it is a string type
* secondly, there is a space in string
* lastly, this word could be written in lower and upper case.

I need to keep in mind these points in order to correctly check examples from the whole dataset

In [ ]:
# Replace None values with np.nan in order to remove rows with missing values
resumes["Job_description"] = resumes["Job_description"].apply(lambda x: x if str(x).lower().replace(' ', '') != "none" and x is not None else np.nan)
resumes["Job_title"] = resumes["Job_title"].apply(lambda x: x if str(x).lower().replace(' ', '') != "none" and x is not None else np.nan)

In [ ]:
# Let's check how does it work
display(resumes.iloc[45])
print('')
display(resumes[resumes["Applicant_ID"] == 1105]["Job_title"])

In [ ]:
print(f"\nShape of resumes dataset: {resumes.shape}")
print(f"\nNumber of unique applicant ID: {resumes.Applicant_ID.nunique()}")

In [ ]:
# Delete rows with missing values in "Job_description" and "Job_title" columns
resumes_clean = resumes.dropna(subset=["Job_description", "Job_title"])

# location, employer and salary won’t be helpful for our recommendation engine, so we’ll remove them as well
resumes_clean = resumes_clean.drop(columns=["Employer.Name", "Salary", "Start.Date", "End.Date",
                                "Can.Contact.Employer", "Created.At", "Updated.At",
                                "City", "State.Name", "State.Code"], axis=1)

print(f"\nShape of resumes dataset: {resumes_clean.shape}")
print(f"\nNumber of unique applicant ID: {resumes_clean.Applicant_ID.nunique()}")

In [ ]:
# Let's visualize the most popular job positions from resumes
applicant_job_title = resumes_clean["Job_title"].value_counts().reset_index()[:15]
applicant_job_title.columns = ["Job_title", "count"]

sns.barplot(
    x="count",
    y="Job_title",
    data=applicant_job_title,
    color="blue",
    palette="hls"
).set(title="Top-15 applicant's job positions");

In [ ]:
display(resumes_clean.head(5))

#### Deleting full duplicates in Job_description columns from Resumes dataset

In [ ]:
"""
Let`s check our Job_description column to duplicates
"""
duplicate = resumes_clean[resumes_clean.duplicated("Job_description")]
 
print("\nDuplicated rows in Job_description: \n")
display(duplicate.iloc[3:10])
print(f"\nNumber of duplicated rows: {duplicate.shape[0]}")

In [ ]:
"""
Delete duplicates by drop_duplicates() method
Create new df resume_dedupped - is cleaned from duplicates
"""
resume_dedupped = resumes_clean.drop_duplicates(subset=["Job_description"])
print(f"Total number of rows after deleting duplicates: {resume_dedupped.shape[0]}")

***

In [ ]:
"""
Pickle datasets that will be used later
"""
vacancies_dedupped.to_pickle("./vacancies.pkl")
resume_dedupped.to_pickle("./resumes.pkl")

# <p style="text-align:center;font-size:100%;">2. Text Preprocessing</p>

Text preprocessing is the practice of cleaning and preparing text data. This is one of the most crucial steps in the process.

Using NLTK, job title and job description are then pre-processed with the following:

* tokenization — convert sentences to words
* converting the text to lower case
* stopwords removal - frequent words which have not any semantic sense
* removing punctuation, numerical values, some extra examples
* lemmatization - convert the word into a root word
    - *using pymystem3 in order to get correct word form based on the context*
* part-of-speech tagging - removing noninformative POS
* vectorization - numerically representation of text (tf-idf vectorization)

At every stage is necessary go through the text manually to try "catch" examples which can definitely show up and hurt the model.

Since after pre-processing, vacancies with short length got even shorter, i.e., less than 20 words. I’ve decided to remove vacancies that are less than 23 words, in order to still have at least 1,000 resumes in the dataset, ensuring that there’s enough text for the model to be trained on.

In [ ]:
# Read the last version
vacancies_temp = pd.read_pickle("./vacancies.pkl")
resume_temp = pd.read_pickle("./resumes.pkl")

In [ ]:
print("Vacancies' dataset: \n")
display(vacancies_temp.head(2))
print(f"\nVacancies' dataset shape: {vacancies_temp.shape}\n")
print("*"*80)
print("\nResumes' dataset: \n")
display(resume_temp.head(2))
print(f"\nResumes' dataset shape: {resume_temp.shape}\n")

In [ ]:
"""
Vacancies dataset
Concat Job title and Job description text into one column
"""
vacancies_temp["Job_title_and_desc"] = vacancies_temp["Job_position"].map(str) + "  " + vacancies_temp["Job_description"]
display(vacancies_temp.head(2))

In [ ]:
"""
Resumes dataset
Last step: to make a resume out of this data, I want to concatenate all job experiences by applicant ID

Usually the information about candidate presents full story of his work experience. It has a sense to merge these "work history" into one.
"""
resume_temp = resume_temp.groupby("Applicant_ID").agg({"Job_title": " ".join,"Job_description": " ".join}).reset_index()
display(resume_temp.head(5))
display(resume_temp.shape)

***

## <p style="text-align:center;font-size:100%;">2.1 Text preprocessing: Part-of-Speech tagging (POS)</p>

Previously I want to do part-of-speech tagging. This step is important for lemmatisation to work, as words which have different meanings depending on part of speech. So, it would be better if POS Tagging implementation is done first.

Also we need to remove not informative words, as prononuns, which is not define job roles.

`What is Part of Speech Tagging?`

Part of Speech Tagging is the process of associating each word in a piece of text with a particular tag, which represents the type of word it is, i.e. proper noun, comparative adjective, interjection etc.

The Universal tagset shown below is a simplified POS tagset:
* CC coordinating conjunction
* CD cardinal digit
* DT determiner
* EX existential there (like: “there is” … think of it like “there exists”)
* FW foreign word
* IN preposition/subordinating conjunction
* JJ adjective ‘big’
* JJR adjective, comparative ‘bigger’
* JJS adjective, superlative ‘biggest’
* LS list marker 1)
* MD modal could, will
* NN noun, singular ‘desk’
* NNS noun plural ‘desks’
* NNP proper noun, singular ‘Harrison’
* NNPS proper noun, plural ‘Americans’
* PDT predeterminer ‘all the kids’
* POS possessive ending parent’s
* PRP personal pronoun I, he, she
* PRP\$ possessive pronoun my, his, hers
* RB adverb very, silently,
* RBR adverb, comparative better
* RBS adverb, superlative best
* RP particle give up
* TO, to go "to" the store
* UH interjection, errrrrrrrm
* VB verb, base form take
* VBD verb, past tense, took
* VBG verb, gerund/present participle taking
* VBN verb, past participle taken
* VBP verb, sing. present, non-3d take
* VBZ verb, 3rd person sing. present takes
* WDT wh-determiner which
* WP wh-pronoun who, what
* WP$ possessive wh-pronoun whose
* WRB wh-adverb where, when

In [ ]:
def text_preprocessing(data):
    """
    - Splitting the text into separate words (token) by capital letter
    (! Be careful with regex in order to save correct version of words writting by capslock)
    Example before:
    "The duties of a janitor, General Cleaner include but not limited to the following:ResponsibilitiesClean restroomsReplenish restroomsEmpty trashEmpty..."
    After:
    "the duties of a janitor,  general  cleaner include but not limited to the following: responsibilities clean restrooms replenish restrooms empty trash empty recycle"
    
    - Converting all the characeters to lower case
    """
    data["Job_title_and_desc"] = data["Job_title_and_desc"].apply(lambda x: re.sub( r"([A-Z][^a-z]*)", r" \1", x))
    data["Job_title_and_desc"] = data["Job_title_and_desc"].str.lower()
    
    return data

In [ ]:
"""
Part-of-speech tagging
"""
def get_pos(data):
    """
    Get column: list of pair - (token, part-of-speech)
    """
    texts = data["Job_title_and_desc"].tolist() #extract the Text column to a list of string
    tagged_texts = pos_tag_sents(map(word_tokenize, texts))
    
    #add the column back to the DataFrame
    data["POS"] = tagged_texts

    return data


def get_informative_token(data):
    """
    Get list of pair with POS which one play a role in define job positions
    Delete noninformative part-of-speech: pronouns, preposition and postposition, etc.
    
    I wanted to remove ADJ, but I couldn't do it because I lost meaningful words: "electrical(ADJ) maintenance technician(ADJ)"
    """
    pos_noninformative = [".", "CC", "CD", "DT", "IN", "LS", "MD", "POS", "PRP",
                          "PRP$", "TO", "UH", "WDT", "WP", "WP$", "WRB"]
    
    data["POS_clean"] = data["POS"].apply(lambda x: [pair for pair in x if pair[0] != "nbsp" and pair[1] not in pos_noninformative])
                
    return data


def get_only_token(data):
    """
    Get column: list of token with meaningful part-of-speech
    """
    data["clean_token"] = data["POS_clean"].apply(lambda x: [word[0] for word in x])
    
    return data


def get_count_of_tokens(data):
    """
    Get column with words number in job description
    """
    data["token_number"] = data["clean_token"].apply(lambda x: len(x))
    
    return data

In [ ]:
"""
Text preprocessing:
- get tokens (separate words)
- convert tokens to lower case
"""
vacancies_temp = text_preprocessing(vacancies_temp)

In [ ]:
"""
Get POS tags
"""
vacancies_temp = get_pos(vacancies_temp)
vacancies_temp = get_informative_token(vacancies_temp)
vacancies_temp = get_only_token(vacancies_temp)
vacancies_temp = get_count_of_tokens(vacancies_temp)

vacancies_temp.head(2)

In [ ]:
"""
Delete tokens with 1 character

I'm not sure that it will be right to delete token with 2 characters, it could be meaningful abbreviation (as RN or LPN from medicine domain)
"""
vacancies_temp["clean_token"] = [[subelt for subelt in elt if len(subelt) > 1] for elt in vacancies_temp["clean_token"]]

#update count_token_number_in_job column
vacancies_temp = get_count_of_tokens(vacancies_temp)
vacancies_temp.head()

In [ ]:
# Put back tokens into one single string for lemmatization
vacancies_temp["clean_job_desc"] = [" ".join(x) for x in vacancies_temp["clean_token"]]
vacancies_temp.head(2)

## <p style="text-align:center;font-size:100%;">2.2 Text preprocessing: Lemmatization</p>

In [ ]:
"""
After getting POS and deleting some of them we can use lemmatization which will return the base form or lemma
"""
wnl = WordNetLemmatizer()
patterns = "[^a-zA-Z \n\.]"

# Use stopwords list from nltk
stopwords_eng = stopwords.words("english")
# add extra stopwords
stopwords_eng.extend(["also", "new", "etc", "part", "time", "hours", "hour",
                      "week", "per", "please", "offer", "part time", "example",
                      "monday", "tuesday", "wednesday", "thursday", "friday",
                     "saturday", "sunday", "pm", "am"])

def lemmatize_sentence(text):
    text = re.sub(patterns, " ", text)
    tokens = []
    
    for token in text.split():
        if token and token not in stopwords_eng:
            token = token.strip()
            token = wnl.lemmatize(token)
            
            tokens.append(token)
            
    return " ".join(tokens) # back to string from list

In [ ]:
print("Before lemmatization:\n", vacancies_temp["clean_job_desc"].iloc[1])
print("\nAfter lemmatization:\n", lemmatize_sentence(vacancies_temp["clean_job_desc"].iloc[1]))

In [ ]:
# Get new column with lemmatize text
vacancies_temp["job_desc_lem"] = vacancies_temp["clean_job_desc"].apply(lemmatize_sentence)
vacancies_temp.head(2)

In [ ]:
# Update token_number_in_job column
vacancies_temp["token_number_after_lem"] = [len(word.split()) for word in vacancies_temp["job_desc_lem"]]
vacancies_temp.head()

***

### Descriptive statistics for word count

In [ ]:
vacancies_temp["token_number_after_lem"].describe()

In [ ]:
"""
Checking the distribution character of word counts

Statistic test to check character of distribution (ab-/normal):
- D'Agostino and Pearson normality test
- alpha = 0.05
"""
# Data visualisation: histogram and boxplot
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 8))

histplot = sns.histplot(
    data=vacancies_temp,
    x="token_number_after_lem",
    bins=25,
    kde=True,
    ax=axes[0]);
histplot.set_title("Distributaion of word counts in job description",  fontsize=16);

boxplot = sns.boxplot(
    data=vacancies_temp,
    x="token_number_after_lem",
    orient="h",
    width=0.9,
    ax=axes[1]);

# Normality test: D'Agostino and Pearson
# Define hypothesis
H0 = "Normal distribution: Data is symmetrically distributed with no skew"
Ha = "NOT normal distribution: The data aren't normally distributed"

# p-value
alpha = 0.05

stat, p = normaltest(vacancies_temp["token_number_after_lem"])
print("Statistics=%.3f, p-value=%.3f" % (stat, p))

# Interpretation
if p > alpha:
    print(f"\n{H0}")
else:
    print(f"\n{Ha}")

Some observations:
* The data aren't normally distributed. There are outliers: one example with more than 2500 words in job description


In [ ]:
"""
Look in details of example with 2616 word counts
"""
vacancies_temp[vacancies_temp["token_number_after_lem"] == 2616]["job_desc_lem"].iloc[0]

This example of job description looks like a little bit strange, as multiple duplication of words. Maybe it has more sense to remove this row from dataset.

**So, what is it optimal number of words in job description?**

A job post should be long enough to be substantive, but short enough to keep a candidate’s attention. The job descriptions that perform best tend to fall between *300 and 660* words total.

*This conclustion is getting from articles (by this link: https://textio.com/blog/how-to-write-a-job-description-in-2020-best-practices-from-half-a-billion-job-postings/28706464272#:~:text=Hit%20the%20sweet%20spot%20for%20word%20count&text=A%20job%20post%20should%20be,300%20and%20660%20words%20total.)*

In our case if we calculate The Interquartile Range (IQR) (calculated as Q3 - Q1) - the middle 50% of the data consists of job description word counts around 95 (Q3 = 150, Q1 = 55). And I guess that it has a sense to remove examples with less than 10 word counts and more than 290 (based on 1.5*IQR values =~290). Because "300-660 words" is about full text of description with all part-of-speech. But I have clean text of job description with meaningful keywords.

Let's look at these examples.

In [ ]:
# Check the number of examples with more than 290 word counts
print("Total number of examples in dataset: ")
display(vacancies_temp.shape[0])

print("\nNumber of examples with more than 290 word counts: ")
display(len(vacancies_temp[vacancies_temp["token_number_after_lem"] > 290]["Job_ID"]))

print("")
display(vacancies_temp[vacancies_temp["token_number_after_lem"] > 290]["job_desc_lem"].iloc[1])

In [ ]:
"""
Check job description with less than 10 keywords
"""
print("Total number of examples in dataset: ")
display(vacancies_temp.shape[0])

print("\nNumber of job vacanies with equals or less than 10 keywords ")
display(vacancies_temp[vacancies_temp["token_number_after_lem"] <= 10]["Job_position"].count())

Some observations:
* The examples of job descriptions with word count *more than 290* is about 7%.
* The examples of job descriptions with word count *less (or equal) than 10* is about 0,3%.

I guess that I can delete them as outliers.

In [ ]:
# Delete job description with less than 10 keywords and more than 290
vacancies_temp = vacancies_temp[(vacancies_temp["token_number_after_lem"] <= 290)&(vacancies_temp["token_number_after_lem"] > 10)]
vacancies_temp.shape[0]

In [ ]:
# Plot a boxplot and histplot of the word counts after deleting Outliers
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 8))

histplot = sns.histplot(
    data=vacancies_temp,
    x="token_number_after_lem",
    color="#60505C",
    bins=25,
    kde=True,
    ax=axes[0]);
histplot.set_title("Distributaion of word counts in job description",  fontsize=16);

boxplot = sns.boxplot(
    data=vacancies_temp,
    x="token_number_after_lem",
    orient="h",
    color="#ff8080",
    width=0.9,
    ax=axes[1]);

# Normality test: D'Agostino and Pearson
# Define hypothesis
H0 = "Normal distribution: Data is symmetrically distributed with no skew"
Ha = "NOT normal distribution: The data aren't normally distributed"

# p-value
alpha = 0.05

stat, p = normaltest(vacancies_temp["token_number_after_lem"])
print("Statistics=%.3f, p-value=%.3f" % (stat, p))

# Interpretation
if p > alpha:
    print(f"\n{H0}")
else:
    print(f"\n{Ha}")

Some observations:
* The data aren't normally distributed. But I guess that we should stop to clean dataset based on word counts in job description, because we can lost meaningful part of information.

In [ ]:
vacancies_temp.head(2)

In [ ]:
# Get the top 20 most common words among all the job descriptions
p_text = [word.split() for word in vacancies_temp["job_desc_lem"]]

# Flaten the list of lists
p_text = [item for sublist in p_text for item in sublist]

# Top 20
top_20 = pd.DataFrame(
    Counter(p_text).most_common(20),
    columns=["word", "frequency"]
)

top_20

In [ ]:
# Plot a bar chart for the top 20 most frequently occuring words
fig = plt.figure(figsize=(20,7))

g = sns.barplot(
    x="word",
    y="frequency",
    data=top_20,
    palette="GnBu_d"
)

g.set_xticklabels(
    g.get_xticklabels(),
    rotation=45,
    fontsize=14
)

plt.yticks(fontsize=14)
plt.xlabel("Words", fontsize=14)
plt.ylabel("Frequency", fontsize=14)
plt.title("Top 20 Words", fontsize=17)


plt.show()

Here are the **top 20 words** by frequency among all the descriptions after processing the text:

* ‘service’, ‘customer', ‘care’, 'experience' and ‘work’ are the top 5

In [ ]:
# Get the number of unique words after processing
num_unique_words = len(set(p_text))
num_unique_words

So, there is a lot of unique words here. I want to check it more: maybe some of the word has low high of frequency.

In [ ]:
"""
Create dataset with frequency of unique words in job description
- Check the number of words with 1 and 5 frequency level
"""
top_all = pd.DataFrame(
    Counter(p_text).items(),
    columns=["word", "frequency"]
).sort_values(["frequency"], ascending=False)

display(top_all.head(2))

print("\nNumber of words with frequency level equals 1:")
display(top_all[top_all["frequency"] == 1]["word"].count())

print("\nNumber of words with frequency level equals or less than 10:")
display(top_all[top_all["frequency"] <= 10]["word"].count())

In [ ]:
top_all[top_all["frequency"] == 1]["word"][:50]

*Type of words with frequency equals 1*

Some observations. There are examples with:
* company's site link or e-mail address ("www.integrityliving.com"),
* merged words (washesdishes, excelmonitor, dutiesrequested)
* abbreviation of word (a.s.,  c.p.r, qtg)
* misspelling of a word (responsable, specifictions, opportunies), etc.

All of these cases could be solved in further study. But in this project at the step of vectorize text data I'll use parameters as min_df and max_df
* max_df is used for removing terms that appear too frequently, also known as "corpus-specific stop words"
* min_df is used for removing terms that appear too infrequently

***

## <p style="text-align:center;font-size:100%;">2.3 Text preprocessing: Vectorization - TF-IDF Vectorizer</p>

In this project I use TF-IDF Vectorization (Term Frequency - Inverse Document Frequency)
* Term Frequency (TF): This summarizes how often a given word appears within a document
* Inverse Document Frequency (IDF): This downscales the words that appear a lot across documents.

TF-IDF Vectorizer takes into account not only how many times a word appears in a document but also how important that word is to the whole corpus.

Also I use *word level TF-IDF*: matrix representing tf-idf scores of every term in different documents.

In [ ]:
"""
Vacancy dataset
update index value after removing rows
"""
vacancies_final = vacancies_temp.reset_index(drop=True)
display(vacancies_final.head(2))
display(vacancies_final.iloc[-2:])

print(f"Shape of final vacancy dataset: {vacancies_final.shape}")

In [ ]:
"""
Get dataset with 5 columns
"""
vacancies_final = vacancies_final[["Job_ID", "Job_position", "Job_description", "Job_title_and_desc", "job_desc_lem"]].iloc[:]
display(vacancies_final.head(2))
print(f"Shape of final vacancy dataset: {vacancies_final.shape}")

In [ ]:
"""
Generate tfidf matrix model for entire corpus
* ‘min_df’ to 5 which will tell the model to ignore words that appear in less than 5 of the job descriptions.
* ‘max_df’ to .95 which will tell the model to ignore words that appear in more than 95% of the job descriptions.

This will help eliminate words that don’t contribute positively to the model.
Also, reducing the number of words (tokens) will help save time on running the model.

But of course it is better to leave the min, max parameters unchanged in order to get high quality of clustering.
But I have not enough machine's processing capabilities in order to run model faster on the defualt parameters.
"""
tfidf_vect = TfidfVectorizer(min_df=5,
                             max_df=0.95
                            )

tfidf = tfidf_vect.fit(vacancies_final["job_desc_lem"]) #fitting the vector
tfidf

***

# <p style="text-align:center;font-size:100%;">3. Resume recommendation based on similarity score</p>

In [ ]:
"""
Text preprocessing of Resumes dataset - we can do it or not.
In this case I run without text preprocessing of New data
"""
# # Concat Job title and Job description text into one column
# resume_temp["Job_title_and_desc"] = resume_temp["Job_title"].map(str) + "  " + resume_temp["Job_description"]

# #Text preprocessing
# resume_temp = text_preprocessing(resume_temp)

# # Get POS tags
# resume_temp = get_pos(resume_temp)
# vacancies_temp = get_informative_token(resume_temp)
# vacancies_temp = get_only_token(resume_temp)
# vacancies_temp = get_count_of_tokens(resume_temp)

# # Delete tokens with 1 character
# resume_temp["clean_token"] = [[subelt for subelt in elt if len(subelt) > 1] for elt in resume_temp["clean_token"]]

# #update count_token_number_in_job column
# resume_temp = get_count_of_tokens(resume_temp)

# # Put back tokens into one single string for lemmatization
# resume_temp["clean_job_desc"] = [" ".join(x) for x in resume_temp["clean_token"]]

# # Get new column with lemmatize text
# resume_temp["job_desc_lem"] = resume_temp["clean_job_desc"].apply(lemmatize_sentence)

# # Update count_token_number_in_job column
# resume_temp["token_number_after_lem"] = [len(word.split()) for word in resume_temp["job_desc_lem"]]

In [ ]:
"""
Pickle datasets that will be used later
"""
vacancies_final.to_pickle("./vacancies_preprocess.pkl")
resume_temp.to_pickle("./resumes_preprocess.pkl")

In [ ]:
# Read the last version
vacancies_final = pd.read_pickle("./vacancies_preprocess.pkl")
resume_final = pd.read_pickle("./resumes_preprocess.pkl")

It's simple way to build Recommender system based on similarity score

I use linear_kernel (not cosine_similarity) because we already get normalized vectors after TF-IDF.

In [ ]:
# Vectorize applicants' job experience
resumes_tfidf = tfidf_vect.transform(resume_final["Job_description"])
resumes_tfidf

In [ ]:
# Choose vacancy id
vacancies_final.head(3)

In [ ]:
"""
Create recommendation of resumes by Job ID
"""
def resumes_to_vacancy(job_id):
    # get vacancy id as an index
    v = job_id
    
    if v in vacancies_final["Job_ID"].tolist():
        index = np.where(vacancies_final["Job_ID"] == v)[0][:]
        vacancy_q = vacancies_final.iloc[index[0]:(index[-1]+1)]
        
        print(f"Information about Vacancy: {v}")
        display(vacancy_q)
        
        # Vectorize vacancy's job description
        vacancy_tfidf = tfidf_vect.transform(vacancy_q["Job_description"])
        
        # compute similarity score
        similarity_score = map(lambda x: linear_kernel(vacancy_tfidf, x), resumes_tfidf)
        output = list(similarity_score)
        
        # getting the job id's of the recommendations
        top = sorted(range(len(output)), key=lambda i: output[i], reverse=True)[:10]
        recommendation = pd.DataFrame(columns = ["Job_ID", "Recommended_Applicant_ID"])
        count = 0
        
        for i in top:
            recommendation.at[count, "Job_ID"] = v
            recommendation.at[count, "Recommended_Applicant_ID"] = resume_final["Applicant_ID"].iloc[i]
            count += 1
            
        # getting the job ids and their data
        nearest_candidates = recommendation["Recommended_Applicant_ID"]
        applicant_recommended = pd.DataFrame(columns = ["Job_ID", "Job_position", "Recommended_Applicant_ID", "Work_experience", "Previous_job"])
        
        for id in nearest_candidates:
            index_resume = resume_final.index[resume_final["Applicant_ID"] == id][0]
            # index = np.where(vacancies_final["Job_ID"] == id)[0][0] - different result
            applicant_recommended.at[count, "Job_ID"] = v
            applicant_recommended.at[count, "Job_position"] = vacancies_final[vacancies_final["Job_ID"] == v]["Job_position"].iloc[0]
            applicant_recommended.at[count, "Recommended_Applicant_ID"] = id
            applicant_recommended.at[count, "Work_experience"] = resume_final["Job_description"][index_resume]
            applicant_recommended.at[count, "Previous_job"] = resume_final["Job_title"][index_resume]
            count += 1
            
        print(f"\nRecommended Applicant_id for Vacancy {v}\n")
        
    else:
        return ("This Job_ID is not in Vacancies' list")
        
    return applicant_recommended

In [ ]:
resumes_to_vacancy(129)

In [ ]:
"""
Create recommendation of vacancies by Applicant ID
"""
def vacancies_for_resume(applicant_id):
    u = applicant_id
    
    if u in resume_final["Applicant_ID"].tolist():
        index = np.where(resume_final["Applicant_ID"] == u)[0][:]
        user_q = resume_final.iloc[index[0]:(index[-1]+1)] # in case if we have more than one rows with applicant_ID
        
        print(f"Information about Applicant: {u}")
        display(user_q)
        
        # Vectorize applicant job experience
        user_tfidf = tfidf_vect.transform(user_q["Job_description"])
        job_tfidf = tfidf_vect.transform(vacancies_final["Job_description"])
        
        # compute similarity score
        similarity_score = map(lambda x: linear_kernel(user_tfidf, x), job_tfidf)
        output2 = list(similarity_score)
        
        # It is necessary to use if-else statement in order to check examples with ONE resume or MORE than ONE resume.
        # Because it needs different action steps to create recommendation for the Applicant
        if len(output2[:][0]) > 1:
            print("\nThis Applicant has more than 1 resume (different job description)\n")
            output2_list = [arr.tolist() for arr in output2] # get list of lists from array
            
            # get pair of similarity value for every job description without extra square brackets
            output2_list_el = []
            for x in output2_list:
                output2_list_el.append([el[0] for el in x])
            
            # getting the job id's of the recommendations - top-20
            top = sorted(range(len(output2_list_el)), key=lambda i: output2_list_el[i], reverse=True)[:10]
            recommendation = pd.DataFrame(columns = ["Applicant_ID", "Recommended_Job_ID"])
            count = 0
            
            for i in top:
                recommendation.at[count, "Applicant_ID"] = u
                recommendation.at[count, "Recommended_Job_ID"] = vacancies_final["Job_ID"].iloc[i]
                count += 1
            # getting the job ids and their data
            nearestjobs = recommendation["Recommended_Job_ID"]
            job_recommended = pd.DataFrame(columns = ["Applicant_ID", "Applicant_job_title", "Recommended_Job_ID", "Job_description", "Job_title"])
            
            for id in nearestjobs:
                index_vacancy = vacancies_final.index[vacancies_final["Job_ID"] == id][0]
                # index = np.where(vacancies_final["Job_ID"] == id)[0][0] - different result
                job_recommended.at[count, "Applicant_ID"] = u
                job_recommended.at[count, "Applicant_job_title"] = resume_final["Job_title"].iloc[index[0]:(index[-1]+1)].tolist()
                job_recommended.at[count, "Recommended_Job_ID"] = id
                job_recommended.at[count, "Job_description"] = vacancies_final["Job_description"][index_vacancy]
                job_recommended.at[count, "Job_title"] = vacancies_final["Job_position"][index_vacancy]
                count += 1
            display(job_recommended.head())
        else:
            print("\nThis Applicant has only 1 resume\n")
            
            # getting the job id's of the recommendations
            top = sorted(range(len(output2)), key=lambda i: output2[i], reverse=True)[:10]
            recommendation = pd.DataFrame(columns = ["Applicant_ID", "Recommended_Job_ID"])
            count = 0
            
            for i in top:
                recommendation.at[count, "Applicant_ID"] = u
                recommendation.at[count, "Recommended_Job_ID"] = vacancies_final["Job_ID"].iloc[i]
                count += 1
            
            # getting the job ids and their data
            nearestjobs = recommendation["Recommended_Job_ID"]
            job_recommended = pd.DataFrame(columns = ["Applicant_ID", "Applicant_job_title", "Recommended_Job_ID", "Job_description", "Job_title"])
            
            for id in nearestjobs:
                index_vacancy = vacancies_final.index[vacancies_final["Job_ID"] == id][0]
                # index = np.where(vacancies_final["Job_ID"] == id)[0][0] - different result
                job_recommended.at[count, "Applicant_ID"] = u
                job_recommended.at[count, "Applicant_job_title"] = resume_final["Job_title"].iloc[index[0]:(index[-1]+1)].tolist()
                job_recommended.at[count, "Recommended_Job_ID"] = id
                job_recommended.at[count, "Job_description"] = vacancies_final["Job_description"][index_vacancy]
                job_recommended.at[count, "Job_title"] = vacancies_final["Job_position"][index_vacancy]
                count += 1
            
            print(f"\nRecommended Job_id for Applicant {u}\n")
    else:
        return ("This Applicant_ID is not in Resumes' list")

    return (job_recommended)

In [ ]:
vacancies_for_resume(14092)

It's look like as a working instrument. But in this case some applicants have more than one resume corresponds to different job position - not the same.
* *customer service representative* - *cashier* - *volunteer services coordinator assistant*

And they describe different side of job.

Moreover, for me, as HR (for example) it is more interesting to get better result in recommendation lists of pull of resumes to vacancy (resumes_to_vacancy). The result as *Founder/CEO Sr. Strategist* / *Car Detailer Manager* / *Server* for *Receptionist* job position looks like very poor.

Let's look at the other possibilities, for example **Clusterization** in order to get more specific recommendation based on the clusters.

***

# <p style="text-align:center;font-size:100%;">4. Clustering </p>
## <p style="text-align:center;font-size:100%;">4.1 Clustering: Preparing dataset </p>

***

## Vacancy data
Run models on the all dataset in order to check accurate clusters number has a lot of time (more than 1.5 hours for model), because we have a lot of features (~14K from matrix shape). That's why I want to use only part of the dataset.

Also I want to get stratified sample which includes subjects from every subgroup (in this case it is Job title), ensuring that it reflects the diversity of examples.

In [ ]:
display(vacancies_final.head(2))
print(f"\nVacancy dataset shape: {vacancies_final.shape}")
print(f"\nNumber of unique Job titles: {vacancies_final.Job_position.nunique()}")

In [ ]:
job_title_vac = vacancies_final["Job_position"].value_counts().reset_index()
job_title_vac.columns = ["Job_position", "count"]

print("\nNumber of job title that occured only ONCE in dataset: \n")
display(job_title_vac[job_title_vac["count"] == 1]["count"].sum())

print("\nNumber of job title with more than 10 frequency: \n")
display(job_title_vac[job_title_vac["count"] >= 10]["count"].sum())

In [ ]:
"""
Add frequecy count as new column to DataFrame
"""
vacancies_final["frequency_job_title"] = vacancies_final.groupby("Job_position")["Job_position"].transform("count")
vacancies_final.head(2)

In [ ]:
"""
Take examples of dataset where frequency of job titles less than 10
"""
vacancies_part = vacancies_final[vacancies_final["frequency_job_title"] >= 10]
display(vacancies_part.head(2))
print(f"\nShape of vacancy dataset for get clusters: {vacancies_part.shape}")

After removing examples from dataset which job title is occured less than 10 times, we get about 20K examples.

Maybe it has more sense to use all of these examples for creating clusters. But I want to ise only part (less number of features -> less time for model training)

In [ ]:
# Get ratio instead of raw numbers using normalize=True
expected_ratio = vacancies_part["Job_position"].value_counts(normalize=True)

# Round and then convert to percentage
expected_ratio = expected_ratio.round(4)*100

# convert to a DataFrame and store in variable 'island_ratios'
# We'll use this variable to compare ratios for samples 
# selected using SRS and Stratified Sampling 
job_title_ratios = pd.DataFrame({'Expected':expected_ratio})
job_title_ratios

In [ ]:
stratified_sample = vacancies_part.groupby("Job_position").apply(lambda x: x.sample(frac=0.80))
display(stratified_sample.head(2))

# Remove the extra index added by groupby()
stratified_sample = stratified_sample.droplevel(0)
display(stratified_sample.head(2))

In [ ]:
# Ratio of selected items by the island
stratified_ratio = stratified_sample["Job_position"].value_counts(normalize=True)

# Convert to percentage
stratified_ratio = stratified_ratio.round(4)*100

# We did stratified sampling. So give it proper name
stratified_ratio.name = "Stratified"

# Add it to the variable job_title_ratios which already has 
# the  expected and SRS proportions 
job_title_ratios = pd.concat([job_title_ratios, stratified_ratio], axis=1)
job_title_ratios

In [ ]:
"""
Vacancy dataset
update index value after removing rows
"""
strat_vacancy_df = vacancies_part.reset_index(drop=True)
display(strat_vacancy_df.head(2))
display(strat_vacancy_df.iloc[-2:])

print(f"Shape of stratified vacancy dataset: {strat_vacancy_df.shape}")

So I got stratified (by Job_title) part of dataset for use it as train for getting clusters number

## <p style="text-align:center;font-size:100%;">4.2 Text clustering: TF-IDF + TruncatedSVD + clustering algorithms</p>

Using clustering can address several known issues in recommendation systems, including increasing the diversity, consistency and reliability of recommendations.

**K-means** is a well-liked unsupervised learning algorithm that organizes data points into groups based on similarities. The algorithm operates by iteratively assigning each data point to its nearest cluster centroid and then recalculating the centroids based on the newly formed clusters.
**Hierarchical agglomerative clustering** is good for this task (text clustering) because documents are non-lattice, non-real valued data that do not live in Euclidean space. Furthermore, this method accounts for the overlapping/concentric structure of topic clusters in document-based data.
**DBSCAN** is a density-based clustering algorithm that groups together points that are closely packed together (i.e., high density) and separates points that are far apart (i.e., low density). But if there are 

In this notebook, we extract TF-IDF features to get a representation of the document at a high level instead, and then apply dimensionality reduction by TruncatedSVD in order to get less features than we have now.

**Steps**:
1) Vectorise our dataset by TfIdfVectorizer()

2) Reduce the dimensionality of the data using TruncatedSVD()
    - *TruncatedSVD is a popular method for dimensionality reduction. However, it works better with sparse data than PCA. This estimator does not center the data before computing the singular value decomposition. In particular, truncated SVD works on term count/tf-idf matrices as returned by the vectorizers*
    
3) Use clustering model - the standard k-means algorithm (actually, a refined variant called k-means++)
    
    - Identifying optimal number of clusters. Our case: the ground truth of a data set is not available. I use Intrinsic Methods to evaluate the clustering quality by examining how well the clusters are separated and how compact the clusters are by:
    
        - Elbow method
            - If the line chart resembles an arm, then the “elbow” (the point of inflection on the curve) is a good indication that the underlying model fits best at that point.
            
        - Silhouette score
            - It is a way to measure how close each point in a cluster to the points in its neighbouring clusters. Silhouette values lies in the range of [-1. 1]. Higher the value is better the cluster configuration. And it is more easy to check number of clusters based on this plot.
            
        - Gap Statistics
            - It standardizes the graph of log(Wk), where Wk is the within-cluster dispersion, by comparing it to its expectation under an appropriate null reference distribution of the data.
            
    - Get clusters in data
    
4) Data visualization: dimensionality reduction by TSNA (non-linear technique)

In [ ]:
#Generate tfidf matrix model for entire corpus - update it with new vocabulary
tfidf_train = TfidfVectorizer(min_df=5, max_df=0.95)

job_part_tfidf = tfidf_train.fit_transform(strat_vacancy_df["job_desc_lem"])
job_part_tfidf

In [ ]:
job_part_tfidf.todense()
print(job_part_tfidf.shape)

Again, we have 19,685 job descriptions and 6,934 unique words.

~7K - It's a lot of features. It's time to reduce dimension

### Dimensionality reduction: Truncated SVD on the TF-IDF matrix to obtain a low-rank approximation of the original matrix

In [ ]:
# Create and run an TSVD with one less than number of features
t0 = time()

tsvd = TruncatedSVD(n_components=job_part_tfidf.shape[1]-1)
X_tsvd = tsvd.fit(job_part_tfidf)

print(f"\nDimensionality reduction with SVD done in {time() - t0:.3f} s")

In [ ]:
# List of explained variances
tsvd_var_ratios = tsvd.explained_variance_ratio_

In [ ]:
"""
Create a function to select the best number of components
"""
def select_n_components(var_ratio, goal_var):
    # Set initial variance explained so far
    total_variance = 0.0
    
    # Set initial number of features
    n_components = 0
    
    # For the explained variance of each feature:
    for explained_variance in var_ratio:
        total_variance += explained_variance #add the explained variance to the total
        n_components += 1 #add one to the number of components
        
        if total_variance >= goal_var: #if we reach our goal level of explained variance
            # End the loop
            break
            
    # Return the number of components
    return n_components

In [ ]:
# Run function
select_n_components(tsvd_var_ratios, 0.95)

Now we get optimal number of components to do dimensionality reduction by TruncatedSVD. Explained variance of the SVD: 94.9%

In [ ]:
# Dimensionality reduction by TruncatedSVD() with optimal number of components
tsvd_train = TruncatedSVD(n_components=2588)
tsvd_matrix = tsvd_train.fit_transform(job_part_tfidf)
explained_variance = tsvd_train.explained_variance_ratio_.sum()

print(f"Shape of matrix: {tsvd_matrix.shape}")
print(f"\nExplained variance of the SVD step: {explained_variance * 100:.1f}%")

### Identifying the optimal number of clusters
- Elbow method
- Silhouette score
- Gap Statistic

In [ ]:
# Create list of inertia and silhouette_score values in different number of clusters
t0 = time()

wcss = []
silhouette = []

# training k_means model
# Fit on data (no need to normalize data, it already is due to TF-IDF)
for i in range(5,101,5):
    kmeans = KMeans(n_clusters=i, init="k-means++", n_init=10, random_state=RANDOM_SEED)
    kmeans.fit(tsvd_matrix)
    wcss.append(kmeans.inertia_)
    silhouette.append(silhouette_score(tsvd_matrix, kmeans.labels_))
    
print(f"\nModel running with 20 different number of clusters done in {time() - t0:.3f} s")

In [ ]:
"""
Elbow method plot
"""
plt.plot(range(5,101,5), wcss)
plt.title("The Elbow Method")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia value")

In [ ]:
"""
Silhouette score plot
"""
plt.plot(range(5,101,5), silhouette)
plt.title("The Silhouette score")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")

In [ ]:
max_score = max(silhouette)
print(f"Optimal number of clusters - {silhouette.index(max_score)*5+5} with silhouette score value: {round(max_score, 3)}")

In [ ]:
# this code from https://www.kaggle.com/code/mallikarjunaj/gap-statistics
"""
Gap Statistic
"""
def optimalK(data, maxClusters):
    """
    Calculates KMeans optimal K using Gap Statistic from Tibshirani, Walther, Hastie
    Params:
        data: ndarry of shape (n_samples, n_features)
        nrefs: number of sample reference datasets to create
        maxClusters: Maximum number of clusters to test for
    Returns: (gaps, optimalK)
    """
    nrefs = 3
    gaps = np.zeros((len(range(5, maxClusters+1, 5)),))
    resultsdf = pd.DataFrame(
        {
            "clusterCount":[],
            "gap":[]
        })
    
    for gap_index, k in enumerate(range(5, maxClusters+1, 5)):

        # Holder for reference dispersion results
        refDisps = np.zeros(nrefs)

        # For n references, generate random sample and perform kmeans getting resulting dispersion of each loop
        for i in range(nrefs):
            
            # Create new random reference set
            randomReference = np.random.random_sample(size=data.shape)
            
            # Fit to it
            km = KMeans(k)
            km.fit(randomReference)
            
            refDisp = km.inertia_
            refDisps[i] = refDisp
            
        # Fit cluster to original data and create dispersion
        km = KMeans(k)
        km.fit(data)
        
        origDisp = km.inertia_

        # Calculate gap statistic
        gap = np.log(np.mean(refDisps)) - np.log(origDisp)

        # Assign this loop's gap statistic to gaps
        gaps[gap_index] = gap
        
        resultsdf.loc[len(resultsdf)] = {"clusterCount": k, "gap": gap}

    return resultsdf

In [ ]:
t0 = time()

gapdf = optimalK(tsvd_matrix, maxClusters=75)
optimal_k = gapdf[gapdf["gap"] == gapdf["gap"].max()]["clusterCount"].iloc[0]

print(f"Optimal k is: {optimal_k}")
print(f"Calculating KMeans optimal K using Gap Statistic done in {time() - t0:.3f} s")

In [ ]:
plt.plot(gapdf.clusterCount, gapdf.gap, linewidth=3)
plt.scatter(gapdf[gapdf.clusterCount == optimal_k].clusterCount, gapdf[gapdf.clusterCount == optimal_k].gap, s=250, c="r")
plt.grid(True)
plt.xlabel("Cluster Count")
plt.ylabel("Gap Value")
plt.title("Gap Values by Cluster Count")
plt.show()

Some observations:
* I can't see a clear elbow, so look at silhoutte score plot or check gap statistic to find the right number of clusters.
* Silhouette score plot:
    * The best score is around 0.1. This value is near 0 and denote overlapping clusters. It indicates that the sample is on or very close to the decision boundary between two neighboring clusters.
    * The highest value of silhouette score is around 45 clusters (0.099).
* Gap statistic:
    * optimal number of clusters is more than 75. Actually is more than 100 too (I get this result when run this function with different maxClusters).

Under conditions of uncertainty, different algorithms can generate competing solutions.

I guess that now we should use as the number of clusters - 45. It's not a lof of and allows us to save some diversity

***

In [ ]:
"""
Before we start to test another clustering algorithms
I want to check one more time how will be better - to use only TF-IDF vectorizer or with TruncatedSVD?
"""
t0 = time()

# run kmeans model with TF-IDF vectors
kmeans_tfidf = KMeans(n_clusters=45, init="k-means++", n_init=10, random_state=RANDOM_SEED)
kmeans_tfidf_clusters = kmeans_tfidf.fit_predict(job_part_tfidf)

print(f"KMeans\non tf-idf vectors: {round(silhouette_score(job_part_tfidf, kmeans_tfidf_clusters), 4)}")
print(f"\nModel running done in {time() - t0:.3f} s")

In [ ]:
t0 = time()

# run MiniBatchKMeans model on TF-IDF vectors
minibatch_tfidf = MiniBatchKMeans(n_clusters=45, n_init=10, random_state=RANDOM_SEED, init_size=1000, batch_size=1000)
minibatch_tfidf_clusters = minibatch_tfidf.fit_predict(job_part_tfidf)

print(f"MiniBatchKMeans\non tf-idf vectors: {round(silhouette_score(job_part_tfidf, minibatch_tfidf_clusters), 4)}")
print(f"\nModel running done in {time() - t0:.3f} s")

In [ ]:
t0 = time()

# run kmeans model with LSA on TF-IDF vectors
kmeans_lsa = KMeans(n_clusters=45, init="k-means++", n_init=10, random_state=RANDOM_SEED)
kmeans_lsa_clusters = kmeans_lsa.fit_predict(tsvd_matrix)

print(f"KMeans with LSA\non tf-idf vectors: {round(silhouette_score(tsvd_matrix, kmeans_lsa_clusters), 4)}")
print(f"\nModel running done in {time() - t0:.3f} s")

In [ ]:
t0 = time()

# run MiniBatchKMeans model with LSA on TF-IDF vectors
minibatch_lsa = MiniBatchKMeans(n_clusters=45, n_init=10, random_state=RANDOM_SEED, init_size=1000, batch_size=1000)
minibatch_lsa_clusters = minibatch_lsa.fit_predict(tsvd_matrix)

print(f"MiniBatchKMeans with LSA\non tf-idf vectors: {round(silhouette_score(tsvd_matrix, minibatch_lsa_clusters), 4)}")
print(f"\nModel running done in {time() - t0:.3f} s")

In [ ]:
tfidf_vs_lsa = pd.DataFrame({
    "Model Name": ["KMeans on TF-IDF", "MiniBatchKMeans on TF-IDF", "KMeans with LSA on TF-IDF", "MiniBatchKMeans with LSA on TF-IDF"],
    "Silhouette score": ["0.0897", "0.0803", "0.0986", "0.09"],
})

display(tfidf_vs_lsa)

* Silhouette score value for clustering model with LSA shows higher result than only on TF-IDF vectors.
* Also the value is higher for K-Means model (k-means++).

### Define the best text clustering model based on the Silhouette score
- K-means (kmeans++)
- Agglomerative clustering
- DBSCAN
    * define optimal values of parameters: eps, min_samples

In [ ]:
"""
Firstly, I need to define optimal value of eps
Use for check part of vacancy dataset too

One way to find the best eps for DBSCAN is to compute the knn, then sort the distances and see where the "knee" is located
"""
neighbors = 6

nbrs = NearestNeighbors(n_neighbors=neighbors).fit(tsvd_matrix)
distances, indices = nbrs.kneighbors(tsvd_matrix)
ns = len(distances[:][0]) # number of samples
distance_desc = sorted(distances[:, ns-1], reverse=True)

# Plotting K-distance Graph
kneedle = KneeLocator(range(1,len(distance_desc)+1),
                      distance_desc, # y values
                      S=1.0, #parameter suggested from paper
                      curve="convex", #parameter from figure
                      direction="decreasing") #parameter from figure

kneedle.plot_knee_normalized()
print(f"\nThe optimal number of eps: {kneedle.knee_y}")

I choose the elbow value, the value of epsilon where there is maximum curvature. Here is around 1.19. **But** when I used this value for train DBSCAN model I've got only ONE cluster. It is not our goal, we should get lower level for eps in order to decrease number of examples in cluster. 

That's why I decided to check one more time different values of eps and minPts parameters with for loop.

In [ ]:
"""
Hyperparameter optimization by for loop
- eps is a distance measure that will be used to locate the points in the neighborhood of any point.
- min_samples is the fewest number of points required to form a cluster
"""
t0 = time()

min_samples = np.arange(5, 21, 5)
eps = np.arange(0.2, 0.6, 0.1) # returns array of ranging from 0.2 to 0.6 with step of 0.1

output_eps = []

for ms in min_samples:
    for ep in eps:
        labels = DBSCAN(min_samples=ms, eps = ep, metric="cosine").fit(tsvd_matrix).labels_
        score = silhouette_score(tsvd_matrix, labels)
        output_eps.append((ms, ep, score))

        
# Get the parameters for best silhouette score
min_samples, eps, score = sorted(output_eps, key=lambda x:x[-1])[-1]
print(f"Best silhouette_score: {round(score, 3)}")
print(f"min_samples: {min_samples}")
print(f"eps: {(round(eps, 1))}")

print(f"\nHyperparameters optimization done in {time() - t0:.3f} s")

The model has 0.034 maximum average silhouette score with epsilon = 0.4 and min_sample = 10.
* min_samples' values [5,10,15,20]
* eps' values [0.2, 0.3, 0.4, 0.5]

So, now I'm ready to compare 3 clustering models based on the **Silhouette score** value

In [ ]:
# apply kmeans algorithm
t0 = time()

kmeans_model = KMeans(n_clusters=45, init="k-means++", n_init=10, random_state=RANDOM_SEED)
kmeans_train = kmeans_model.fit(tsvd_matrix)
print(f"\nModel running done in {time() - t0:.3f} s")

In [ ]:
# apply agglomerative algorithm
t0 = time()

agglo_model = AgglomerativeClustering(linkage="ward", n_clusters=45)
agglomerative_clusters = agglo_model.fit_predict(tsvd_matrix)
print(f"\nModel running done in {time() - t0:.3f} s")

In [ ]:
# apply DBSCAN algorithm: When dealing with texts, the distance metric to be used is cosine instead of "euclidean"
t0 = time()

model_dbscan = DBSCAN(eps=0.4, min_samples=10, metric="cosine")
dbscan_clusters = model_dbscan.fit_predict(tsvd_matrix)

print(f"\nModel running done in {time() - t0:.3f} s")

In [ ]:
def silhouette_method(df, algo, pred_clusters):
  print("=================================================================================")
  print(f"Clustering: {algo}  Silhouette score: {silhouette_score(df, pred_clusters)}")


silhouette_method(tsvd_matrix, "KMeans", kmeans_model.labels_)
silhouette_method(tsvd_matrix, "Agglomerative", agglomerative_clusters)
silhouette_method(tsvd_matrix, "DBSCAN", dbscan_clusters)
print("=================================================================================")

The highest value of *silhouette score* belongs to K-means. I can choose it for further work.

In [ ]:
"""
Save kmeans model
"""
with open("./kmeans_model.pickle", "wb") as pkl_file:
    pickle.dump(kmeans_train, pkl_file)

## For, loading the final model
# with open("./kmeans_model.pickle", "rb") as pkl_file:
#     model_kmeans = pickle.load(pkl_file)

***

### Get clusters based on the K-means algorithm

In the previous step I run the clustering algorithms on some part of the dataset (stratified). The reason of this decesion is simple: using full dataset took a lot of time at the stage of checking optimal number of clusters (hours and hours...).

Now I have optimal number of clusters - 45 and trained K-Means model. My next step is to predict clusters of the full dataset in order to use it for recommendation engine. I need to get the same format/ shape of full data as fitted data for kmeans model. That's why I use TF-IDF vectoriser and TruncatedSVD were trained in the previous step.

In [ ]:
# Let's vectorize our full data: TfIdf -> TruncatedSVD
t0 = time()

job_tfidf = tfidf_train.transform(vacancies_final["Job_title_and_desc"])
vacancy_svd = tsvd_train.transform(job_tfidf)
display(vacancy_svd.shape)
print(f"\nVectorizing and dimension reduction done in {time() - t0:.3f} s")

In [ ]:
"""
Let's run fitted kmeans++ model with 55 clusters
"""
t0 = time()

kmeans_cluster = (kmeans_train.predict(vacancy_svd)).tolist()# fit the vectors using KMeans
print(f"\nModel running done in {time() - t0:.3f} s")

In [ ]:
# examine the cluster shape
kmeans_train.cluster_centers_.shape

We can see by looking at the shape of the cluster_centers we have 45 clusters that have a dimension space of 2,588.

Next I'll create a new column in vacancies dataframe that has the value for the cluster that each response was assigned to. This allows to do some investigating, which I will do next.

In [ ]:
# assign labels as an additional dataframe column
vacancies_final["kmeans_cluster"] = kmeans_cluster

# Display number of documents in each cluster
vacancies_final["kmeans_cluster"].value_counts()

In [ ]:
# Print a sample of 3 documents that belong to 3 clusters
for c in vacancies_final["kmeans_cluster"].value_counts().index[:3] :
    print("CLUSTER ", c , " :")
    print("----")
    for d in vacancies_final.loc[vacancies_final["kmeans_cluster"]==c,:].sample(3)["job_desc_lem"]:
        print(d)
        print()
    print("-----------")

In [ ]:
# Wordcloud for the 5 first clusters
wd = wordcloud.WordCloud()

for c in vacancies_final["kmeans_cluster"].value_counts().index[:5] :
    print("CLUSTER ", c)
    texts = " ".join(vacancies_final.loc[vacancies_final["kmeans_cluster"]==c,"job_desc_lem"])
    cloud = wd.generate(texts)
    plt.imshow(cloud)
    plt.show()
    print('-----------')

In [ ]:
total_count = vacancies_final["kmeans_cluster"].count()

cluster = 25300
total_responses = total_count

round(cluster/total_responses, 2)

About 46% of the job descriptions are assigned to cluster 5 with high value of examples.

If we will increase our k surely some of those would have gone into the additional clusters, but in general a lot of those responses are just difficult to categorize by only focusing on individual words.

### Investigating the Clusters

In [ ]:
"""
Let's look at cluster 11
We can see here that in general cluster 11 include job description of job position corresponds to Customer Service Representative
"""
display(vacancies_final[vacancies_final["kmeans_cluster"] == 11][["job_desc_lem", "Job_position"]].iloc[:10])
print("*"*80)
display(vacancies_final[vacancies_final["kmeans_cluster"] == 11][["job_desc_lem", "Job_position"]].iloc[-10:])

In [ ]:
"""
Let's look at cluster 13

Cluster 13 makes a lot of references to workers from Medicine domain
"""
display(vacancies_final[vacancies_final["kmeans_cluster"] == 13][["job_desc_lem", "Job_position"]].iloc[:10])
print("*"*80)
display(vacancies_final[vacancies_final["kmeans_cluster"] == 13][["job_desc_lem", "Job_position"]].iloc[-10:])

In [ ]:
"""
Let's look at cluster 9

Cluster 9 makes a lot of references to Account Receivable Specialists
"""
display(vacancies_final[vacancies_final["kmeans_cluster"]==9][["job_desc_lem", "Job_position"]].iloc[:10])
print("*"*80)
display(vacancies_final[vacancies_final["kmeans_cluster"]==9][["job_desc_lem", "Job_position"]].iloc[-10:])

In [ ]:
"""
Let's look at cluster 33

Cluster 33 makes a lot of references to workers on Kitchen Positions
"""
display(vacancies_final[vacancies_final["kmeans_cluster"]==33][["job_desc_lem", "Job_position"]].iloc[:10])
print("*"*80)
display(vacancies_final[vacancies_final["kmeans_cluster"]==33][["job_desc_lem", "Job_position"]].iloc[-10:])

### Data visualization: dimensionality reduction with TSNA

In [ ]:
t0 = time()

tsne = TSNE(verbose=1, perplexity=50)  # Changed perplexity from 100 to 50 per FAQ
job_tsne = tsne.fit_transform(vacancy_svd)

print(f"\nDimensionality reduction with TSNE done in {time() - t0:.3f} s")

In [ ]:
clusters_kmeans = vacancies_final["kmeans_cluster"]

# sns settings
sns.set(rc={"figure.figsize":(13,9)})

# colors
palette = sns.hls_palette(20, l=.4, s=.9)

# plot
sns.scatterplot(x=job_tsne[:,0], y=job_tsne[:,1], hue=clusters_kmeans, legend=False, palette=palette)
plt.title("t-SNE with Kmeans Labels")
plt.show()

The labeled plot gives better insight into how the job descriptions are grouped.

Now there are cases where the colored labels are spread out on the plot. This is a result of t-SNE and k-means finding different connections in the higher dimensional data. The topics of these job descriptions often intersect so it was hard to cleanly separate them.

Also, I see that there are ONE big topic. I guess that it presents job examples from the same industry: Office workers. It could be administrative assistant, customer service representative, etc. Yes these job positions are different, but anyway they are closer to each other than to job position from Medicine or Science domain.

Based on these findings, maybe it has more sense to use less number of clusters. I want to use for this purpose Topic modeling technique - Non-Negative Matrix Factorization (NMF). Let's go to see how does it work.

***

## <p style="text-align:center;font-size:100%;">Recommendation based on TF-IDF + KMeans algorithm</p>

In [ ]:
resume_for_kmeans = resume_final.copy()
resume_for_kmeans.head(2)

In [ ]:
"""
Predict kmeans model and get clusters for full resume dataset
"""
# Vectorize
resume_tfidf = tfidf_train.transform(resume_for_kmeans["Job_description"])
resume_svd = tsvd_train.transform(resume_tfidf)

# Create clusters
applicant_cluster = (kmeans_train.predict(resume_svd)).tolist()

resume_for_kmeans["kmeans_cluster"] = applicant_cluster
display(resume_for_kmeans.head(2))

In [ ]:
"""
Create recommendation of resumes by Job ID
"""
def resumes_to_vacancy_kmeans(job_id):
    # get vacancy id as an index
    v = job_id
    
    if v in vacancies_final["Job_ID"].tolist():
        index = np.where(vacancies_final["Job_ID"] == v)[0][:]
        vacancy_q = vacancies_final.iloc[index[0]:(index[-1]+1)]
        cluster_num = vacancies_final[vacancies_final["Job_ID"] == v]["kmeans_cluster"].iloc[0]
        
        print(f"Information about Vacancy: {v}")
        display(vacancy_q)
        print(f"This vacancy belongs to cluster: {cluster_num}")
        
        # Get resume with the same topic
        pull_of_resumes = resume_for_kmeans[resume_for_kmeans["kmeans_cluster"] == cluster_num]
        
        # Vectorize vacancy's job description and resume
        vacancy_tfidf = tfidf_train.transform(vacancy_q["Job_description"])
        resume_tfidf = tfidf_train.transform(pull_of_resumes["Job_description"])
        
        # compute similarity score
        similarity_score = map(lambda x: linear_kernel(vacancy_tfidf, x), resume_tfidf)
        output = list(similarity_score)
        
        # getting the job id's of the recommendations
        top = sorted(range(len(output)), key=lambda i: output[i], reverse=True)[:10]
        recommendation = pd.DataFrame(columns = ["Job_ID", "Recommended_Applicant_ID"])
        count = 0
        
        for i in top:
            recommendation.at[count, "Job_ID"] = v
            recommendation.at[count, "Recommended_Applicant_ID"] = pull_of_resumes["Applicant_ID"].iloc[i]
            count += 1
            
        # getting the job ids and their data
        nearest_candidates = recommendation["Recommended_Applicant_ID"]
        applicant_recommended = pd.DataFrame(columns = ["Job_ID", "Job_position", "Recommended_Applicant_ID", "Work_experience", "Previous_job"])
        
        for id in nearest_candidates:
            index_resume = pull_of_resumes.index[pull_of_resumes["Applicant_ID"] == id][0]
            # index = np.where(vacancies_final["Job_ID"] == id)[0][0] - different result
            applicant_recommended.at[count, "Job_ID"] = v
            applicant_recommended.at[count, "Job_position"] = vacancies_final[vacancies_final["Job_ID"] == v]["Job_position"].iloc[0]
            applicant_recommended.at[count, "Recommended_Applicant_ID"] = id
            applicant_recommended.at[count, "Work_experience"] = pull_of_resumes["Job_description"][index_resume]
            applicant_recommended.at[count, "Previous_job"] = pull_of_resumes["Job_title"][index_resume]
            count += 1
            
        print(f"\nRecommended Applicant_id for Vacancy {v}\n")
        
    else:
        return ("This Job_ID is not in Vacancies' list")
        
    return applicant_recommended

In [ ]:
resumes_to_vacancy_kmeans(129)

This recommendation looks better than the previous one based only similarity score value.

***

## <p style="text-align:center;font-size:100%;">4.3 Topic modeling: Non-Negative Matrix Factorization (NMF)</p>

NMF is an unsupervised technique so there are no labeling of topics that the model will be trained on. The way it works is that, NMF decomposes (or factorizes) high-dimensional vectors into a lower-dimensional representation. These lower-dimensional vectors are non-negative which also means their coefficients are non-negative.

Using the original matrix (A), NMF will give two matrices (W and H). W is the topics it found and H is the coefficients (weights) for those topics. In other words,
* A is job descriptions by words (original),
* H is job descriptions by topics,
* W is topics by words.

NMF will modify the initial values of W and H so that the product approaches A until either the approximation error converges or the max iterations are reached.

In our case, the high-dimensional vectors are going to be tf-idf weights.

### Automatically Selecting the Best Number of Topics

**Coherence Score**

To evaluate the best number of topics, I can use the coherence score. In general it measures the relative distance between words within a topic.

There are a few different types of coherence score with the two most popular being c_v and u_mass.
* c_v is more accurate while u_mass is faster

We’ll be using c_v here which ranges from 0 to 1 with 1 being perfectly coherent topics.

In sklearn’s implementation of NMF it is possible to use tf-idf weights. However, sklearn’s NMF implementation does not have a coherence score.

Therefore, I’ll use gensim to get the best number of topics with the coherence score and then use that number of topics for the sklearn implementation of NMF.

In [ ]:
# Create column: job_description after lemmatization -> token
vacancies_final["token"] = vacancies_final["job_desc_lem"].apply(lambda x: x.split())
vacancies_final.head(2)

In [ ]:
# Use Gensim's NMF to get the best num of topics via coherence score
texts = vacancies_final["token"]

# Create a dictionary
# In gensim a dictionary is a mapping between words and their integer id
dictionary = Dictionary(texts)

# Filter out extremes to limit the number of features
dictionary.filter_extremes(
    no_below=3,
    no_above=0.85,
    keep_n=20000
)

In [ ]:
# Create the bag-of-words format (list of (token_id, token_count))
corpus = [dictionary.doc2bow(text) for text in texts]

In [ ]:
# Create a list of the topic numbers we want to try
topic_nums = list(np.arange(5, 75+1, 5))

In [ ]:
# Run the nmf model and calculate the coherence score
# for each number of topics
t0 = time()

coherence_scores = []

for num in topic_nums:
    nmf = Nmf(
        corpus=corpus,
        num_topics=num,
        id2word=dictionary,
        chunksize=2000,
        passes=5,
        kappa=.1,
        minimum_probability=0.01,
        w_max_iter=300,
        w_stop_condition=0.0001,
        h_max_iter=100,
        h_stop_condition=0.001,
        eval_every=10,
        normalize=True,
        random_state=RANDOM_SEED
    )
    
    # Run the coherence model to get the score
    cm = CoherenceModel(
        model=nmf,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v"
    )
    
    coherence_scores.append(round(cm.get_coherence(), 5))
    
print(f"\nCalculating Coherence score for each number of topics done in {time() - t0:.3f} s")

In [ ]:
len(coherence_scores)

In [ ]:
# Get the number of topics with the highest coherence score
scores = list(zip(topic_nums, coherence_scores))
best_num_topics = sorted(scores, key=itemgetter(1), reverse=True)[0][0]

In [ ]:
# Plot the results
fig = plt.figure(figsize=(15, 7))

plt.plot(
    topic_nums,
    coherence_scores,
    linewidth=3,
    color="#4287f5"
)

plt.xlabel("Topic Num", fontsize=14)
plt.ylabel("Coherence Score", fontsize=14)
plt.title(f"Coherence Score by Topic Number - Best Number of Topics: {best_num_topics}", fontsize=18)
plt.xticks(np.arange(5, max(topic_nums) + 1, 5), fontsize=12)
plt.yticks(fontsize=12)

plt.show()

For the number of topics to try out, I choose a range of 5 to 100 with a step of 5. This just comes from some trial and error, the number of job descriptions and average length of the job descriptions.

*Running too many topics will take a long time*

**15 is the number of topics** that returned the highest coherence score (more than .57) and it drops off pretty fast after that.

### Summarizing Topics

Another challenge is summarizing the topics.

The best solution here would to have a human go through the texts and manually create topics. This is obviously not ideal.

Another option is to use the words in each topic that had the highest score for that topic and them map those back to the feature names. We’ll use top 10 words.

In [ ]:
tfidf_nmf = tfidf_vect.transform(vacancies_final["job_desc_lem"]) # tfidf_vect fitted on full vacancies dataset (after preprocessing)
display(tfidf_nmf)

In [ ]:
best_num_topics = 15

# Save the feature names for later to create topic summaries
tfidf_fn = tfidf_vect.get_feature_names_out()

# Run the nmf model
nmf = NMF(
    n_components=best_num_topics,
    init="nndsvd",
    max_iter=500,
    l1_ratio=0.0,
    solver="cd",
    alpha_W=0.0,
    tol=1e-4,
    random_state=RANDOM_SEED
).fit(tfidf_nmf)

In [ ]:
def top_words(topic, n_top_words):
    """
    Function to get top words in each topic
    """
    return topic.argsort()[:-n_top_words - 1:-1] 


def topic_table(model, feature_names, n_top_words):
    """
    Function to get df with topics and top words
    """
    topics = {}
    for topic_idx, topic in enumerate(model.components_):
        t = (topic_idx)
        topics[t] = [feature_names[i] for i in top_words(topic, n_top_words)]
        
    return pd.DataFrame(topics)

In [ ]:
"""
Use the top words for each cluster by tfidf weight to create 'topics'
"""

# Getting a df with each topic by document
docweights = nmf.transform(tfidf_vect.transform(vacancies_final["job_desc_lem"]))

n_top_words = 10

topic_df = topic_table(
    nmf,
    tfidf_fn,
    n_top_words
).T

# Cleaning up the top words to create topic summaries
topic_df["topics"] = topic_df.apply(lambda x: [", ".join(x)], axis=1) # Joining each word into a list
topic_df["topics"] = topic_df["topics"].apply(lambda x: list(set(x)))  # Removing duplicate words
topic_df["topics"] = topic_df["topics"].str[0] # Removing the list brackets

topic_df.head()

In [ ]:
# Create a df with only the created topics and topic num
topic_df = topic_df["topics"].reset_index()
topic_df.columns = ["topic_nmf", "topics"]

topic_df

* Topic 0: work for students (maybe it is about intern level positions)
* Topics 1 and 14 includes words related to the field of Medicine
* Topic 3: words from this topic describe Office workers
* Topic 6: words from this topic describe Dаta entry specialist
* Topic 5, 7: words from these topics describe Front Desk Representative (CSR)
* Topic 9: words from this topic describe Retail positions
* Topics 2, 4, 8, 10-12 includes words related to the field of Finance
* Topic 13: words from this topic describe Kitchen positions

In [ ]:
# Creating a temp df with the job_id and topic num to join on
job_id = vacancies_final["Job_ID"].tolist()

df_temp = pd.DataFrame({
    "Job_ID": job_id,
    "topic_nmf": docweights.argmax(axis=1)
})

df_temp["topic_nmf"].nunique()

In [ ]:
# Merging to get the topic num with url
merged_topic = df_temp.merge(
    topic_df,
    on="topic_nmf",
    how="left"
)

merged_topic

In [ ]:
# Merging with the original df
vacancies_final = pd.merge(
    vacancies_final,
    merged_topic,
    on="Job_ID",
    how="left"
)

vacancies_final.head()

In [ ]:
vacancies_final["topic_nmf"].value_counts(normalize=True)

* Cluster 0 is the most "popular": 
* Cluster 8 is the least "popular":
* At the first glance, the distribution of examples on different topics looks close to the real one.

In [ ]:
# Showing the 3 matrices we get with nmf
A = tfidf_vect.transform(vacancies_final["job_desc_lem"])
W = nmf.components_
H = nmf.transform(A)

print('A = {} x {}'.format(A.shape[0], A.shape[1]))
print('W = {} x {}'.format(W.shape[0], W.shape[1]))
print('H = {} x {}'.format(H.shape[0], H.shape[1]))

* A is job description by words (original) = 55,153 documents * 14,206 unique words
* H is job description by topics = 55,153 documents * 15 topics
* W is topics by words = 15 topics * 14,206 unique words

So assuming 55,153 documents, 14,206 words and 15 topics we would get the following 3 matrices

## Exploring topics

In [ ]:
# Office workers, Support services
display(vacancies_final[vacancies_final["topic_nmf"] == 0]["Job_position"].iloc[:10])
print("*"*80)
display(vacancies_final[vacancies_final["topic_nmf"] == 0]["Job_position"].iloc[-10:])

In [ ]:
# Retail (Outliers as Sushi Chef)
display(vacancies_final[vacancies_final["topic_nmf"] == 5]["Job_position"].iloc[:10])
print("*"*80)
display(vacancies_final[vacancies_final["topic_nmf"] == 5]["Job_position"].iloc[-10:])

In [ ]:
# Medicine
display(vacancies_final[vacancies_final["topic_nmf"] == 14]["Job_position"].iloc[:10])
print("*"*80)
display(vacancies_final[vacancies_final["topic_nmf"] == 14]["Job_position"].iloc[-10:])

In [ ]:
"""
Save nmf model
"""
with open("./nmf_model.pickle", "wb") as pkl_file:
    pickle.dump(nmf, pkl_file)

## For, loading the final model
# with open("./nmf_model.pickle", "rb") as pkl_file:
#     model_nmf = pickle.load(pkl_file)

## <center> Recommendation based on TF-IDF + NMF topic modeling

In [ ]:
resume_for_nmf = resume_final.copy()
resume_for_nmf.head(2)

In [ ]:
# Predicting the topic for applicant job description
new_texts = resume_for_nmf["Job_description"]

# Transform the new data with the fitted models
tfidf_resume = tfidf_vect.transform(new_texts)
resume_nmf = nmf.transform(tfidf_resume)

In [ ]:
# Get the top predicted topic
predicted_topics = [np.argsort(each)[::-1][0] for each in resume_nmf]

# Add to the df
resume_for_nmf["pred_topic_num"] = predicted_topics

resume_for_nmf.head()

In [ ]:
resume_for_nmf["pred_topic_num"].value_counts()

In [ ]:
# Join with the original df to get the topic summary
resume_for_nmf = pd.merge(
    resume_for_nmf,
    vacancies_final[["topic_nmf", "topics"]],
    left_on="pred_topic_num",
    right_on="topic_nmf",
    how="inner"
).drop_duplicates().drop(["topic_nmf"], axis=1)

resume_for_nmf.head()

In [ ]:
resume_for_nmf = resume_for_nmf.reset_index(drop=True)
display(resume_for_nmf.iloc[-10:])
print("")
display(resume_for_nmf.shape)

In [ ]:
# Server, CSR
resume_for_nmf[resume_for_nmf["pred_topic_num"] == 5]["Job_title"].iloc[-15:]

In [ ]:
"""
Create recommendation of resumes by Job ID
"""
def resumes_to_vacancy_nmf(job_id):
    # get vacancy id as an index
    v = job_id
    
    if v in vacancies_final["Job_ID"].tolist():
        index = np.where(vacancies_final["Job_ID"] == v)[0][:]
        vacancy_q = vacancies_final.iloc[index[0]:(index[-1]+1)]
        topic_num = vacancies_final[vacancies_final["Job_ID"] == v]["topic_nmf"].iloc[0]
        
        print(f"Information about Vacancy: {v}")
        display(vacancy_q)
        print(f"This vacancy belongs to topic: {topic_num}")
        
        # Get resume with the same topic
        pull_of_resumes = resume_for_nmf[resume_for_nmf["pred_topic_num"] == topic_num]
        
        # Vectorize vacancy's job description and resume
        vacancy_tfidf = tfidf_vect.transform(vacancy_q["Job_description"])
        resume_tfidf = tfidf_vect.transform(pull_of_resumes["Job_description"])
        
        # compute similarity score
        similarity_score = map(lambda x: linear_kernel(vacancy_tfidf, x), resume_tfidf)
        output = list(similarity_score)
        
        # getting the job id's of the recommendations
        top = sorted(range(len(output)), key=lambda i: output[i], reverse=True)[:10]
        recommendation = pd.DataFrame(columns = ["Job_ID", "Recommended_Applicant_ID"])
        count = 0
        
        for i in top:
            recommendation.at[count, "Job_ID"] = v
            recommendation.at[count, "Recommended_Applicant_ID"] = pull_of_resumes["Applicant_ID"].iloc[i]
            count += 1
            
        # getting the job ids and their data
        nearest_candidates = recommendation["Recommended_Applicant_ID"]
        applicant_recommended = pd.DataFrame(columns = ["Job_ID", "Job_position", "Recommended_Applicant_ID", "Work_experience", "Previous_job"])
        
        for id in nearest_candidates:
            index_resume = pull_of_resumes.index[pull_of_resumes["Applicant_ID"] == id][0]
            # index = np.where(vacancies_final["Job_ID"] == id)[0][0] - different result
            applicant_recommended.at[count, "Job_ID"] = v
            applicant_recommended.at[count, "Job_position"] = vacancies_final[vacancies_final["Job_ID"] == v]["Job_position"].iloc[0]
            applicant_recommended.at[count, "Recommended_Applicant_ID"] = id
            applicant_recommended.at[count, "Work_experience"] = pull_of_resumes["Job_description"][index_resume]
            applicant_recommended.at[count, "Previous_job"] = pull_of_resumes["Job_title"][index_resume]
            count += 1
            
        print(f"\nRecommended Applicant_id for Vacancy {v}\n")
        
    else:
        return ("This Job_ID is not in Vacancies' list")
        
    return applicant_recommended

In [ ]:
resumes_to_vacancy_nmf(129)

So, our HR specialist has a pull of top-10 resumes corresponds to vacancy ("Receptionist"). His/ her task is to organize meeting to check candidate's skills.

***

### 45 clusters VS 15 topics
I get 45 clusters for K-Means model and 15 topics from NMF topic modeilng. Let's look at the content of our clusters in topics.

In [ ]:
clusters_in_topics = vacancies_final.groupby(["topic_nmf", "kmeans_cluster"])["Job_position"].unique()
clusters_in_topics[-50:]

I see that our topics include clusters with the similar job positions (from the same field). That's why I assume that 15 topics will be enough.

My last step is to create a dataframe with recommended pull of resumes (applicants' id).

***

### Create a dataframe with recommended resume

In [ ]:
# get result the best recommendations based on similarity score as pairwise_distances/ or cosine_similarity
recs = pairwise_distances(docweights, resume_nmf, metric="cosine").argsort()

# create empty lists
top_rec_app_ids = []
top_rec_app_positions = []
top_rec_app_resumes = []
top_10_rec_app_ids = []

# get the information about the top recommended resumes to each Vacancies
for rec in recs:
    doc_num = rec[0]
    top_rec_app_ids.append(resume_final["Applicant_ID"].loc[doc_num])
    top_rec_app_positions.append(resume_final["Job_title"].loc[doc_num])
    top_rec_app_resumes.append(resume_final["Job_description"].loc[doc_num])
    top_10_rec_app_ids.append(list(rec[0:10]))

# create dict with updated information about recommended resumes to each Vacancies
rec_dict = {
    "Job_ID": vacancies_final["Job_ID"].tolist(), 
    "Jo_position": vacancies_final["Job_position"].tolist(),
    "top_rec_app_id": top_rec_app_ids,
    "top_rec_app_position": top_rec_app_positions,
    "top_rec_app_resume": top_rec_app_resumes, 
    "top_10_rec_app_ids": top_10_rec_app_ids
}

# create dataframe with recommended resumes
rec_df = pd.DataFrame.from_dict(rec_dict)
rec_df.iloc[:10]

***

## <center> **FURTHER RESEARCH**

On this project I've tried 3 approaches to build Candidate resume recommendation engine:
- Recommender System based on the similarity score
- Clustering algorithms + RS based on the similarity score
    - K-means, Agglomerative clustering, DBSCAN with optimal number of clusters equals 45
- Topic modeling + RS based on the similarity score
    - NMF with optimal number of clusters equals 15

Based on the quality of recommendation (manually checked) I can conclude that approach with Clustering/ Topic modeling step plays a meaningful role in Recommender system building.


As a **further research step**, I can suggest next points:
- Data:
    * Get more informative dataframe/ use web scraping to get resumes and vacancies full of information in the next segments: education level, industry, requirenments. It's possible to use *industry* feature as class label and build recommender system based on classification models (LSV, NB, etc. + ensembles). Also if we have any *requirenments* in vacany's postings it will be useful to get information about total years of applicant's work experience from start and end date, for example.
    * Manually check the same job positions with different variation of title name and replace it to one "more popular" example
- Text preprocessing:
    * Expanding abbreviations
    * Use the other type of regex to clean text from information about company's site, merged words, etc.
    * Using another type of algorithm for vectorization (word2vec - Skip-Gram or BERT)
    * Using another method to get part-of-speech (with high level of correct tagging).
- Model:
    * Use topic modeling as LSA or LDA. These methods are more suited in domains where data is in "semantic units" like words.
    * Create classification models in order to predict new resumes to corresponded topics.
    * Explore the other approaches of Recommender System in case when we have no any labeled data.

***

## BONUS: sentence_transformers

We can use sentence_transformers in order to get groups of job_titles manually based on the benchmark.

In [ ]:
!pip install sentence_transformers

import torch
from sentence_transformers import SentenceTransformer

In [ ]:
"""
Sentence-transformers model
It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search
"""
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
def cos_sim(a, b):
    # From https://github.com/UKPLab/sentence-transformers/blob/master/sentence_transformers/util.py#L31
    """
    Computes the cosine similarity cos_sim(a[i], b[j]) for all i and j.
    :return: Matrix with res[i][j]  = cos_sim(a[i], b[j])
    """
    if not isinstance(a, torch.Tensor):
        a = torch.tensor(a)

    if not isinstance(b, torch.Tensor):
        b = torch.tensor(b)

    if len(a.shape) == 1:
        a = a.unsqueeze(0)

    if len(b.shape) == 1:
        b = b.unsqueeze(0)

    a_norm = torch.nn.functional.normalize(a, p=2, dim=1)
    b_norm = torch.nn.functional.normalize(b, p=2, dim=1)
    return torch.mm(a_norm, b_norm.transpose(0, 1))

In [ ]:
vacancies_test = job_description[["Job.ID", "Position"]]
vacancies_test = vacancies_test.dropna(subset=["Position"])

display(vacancies_test.head())
display(vacancies_test.shape)

In [ ]:
vacancies_test["cnt_characters_in_title"] = vacancies_test["Position"].apply(lambda x: len(x))
vacancies_test.head()

In [ ]:
job_titles = []

# delete examples with less than 1 symbol in job title
for el in vacancies_test.loc[vacancies_test["cnt_characters_in_title"] > 1]["Position"].unique():
    job_titles.append(el)

curr_titles = np.array(sorted(job_titles, reverse=True))

curr_embed = model.encode(
        curr_titles,
        batch_size=64,
        show_progress_bar=True
)

In [ ]:
matrix = cos_sim(curr_embed, curr_embed)
len(matrix)

In [ ]:
for i, row in enumerate(matrix):
    break

for el in np.argsort(-row):
    if matrix[i][el] < 0.55: break
    print(curr_titles[el], round(matrix[i][el].item(), 2))

In [ ]:
# Pull of job positions on RESTAURANT area
lst1 = """
★ RESTAURANT / BAR / RETAIL / CUSTOMER SERVICE EXPERIENCE WANTED ★ 1.0
RESTAURANT / BAR / RETAIL / CUSTOMER SERVICE EXPERIENCE WANTED! 0.8
RESTAURANT / BAR / RETAIL / CUSTOMER SERVICE EXPERIENCE WANTED 0.8
Restaurant or Hospitality Experience Wanted for Sales & Marketing Firm 0.69
Restaurant Customer Service 0.64
Restaurant Cook - Prep - Bartender - Barback - Cashier - Server 0.61
Servers, Cashiers, Bar & Greeters 0.6
Hiring Restaurant Positions - Servers - Host Staff - Bartenders 0.59
Hiring Restaurant Positions - Servers - Bartenders 0.59
Restaurant / Retail / Bar / Athletic Experience Needed! 0.59
Hiring Restaurant Positions - Servers -  Bartenders - Host Staff 0.59
Hiring Restaurant Positions - Servers - Bartenders - Host Staff 0.59
Hiring Restaurant Positions - Servers - Bussers - Bartenders 0.58
Hiring Restaurant Positions - Servers - Cooks - Bartenders 0.58
Hiring Restaurant Positions - Servers - Hosts - Bartenders 0.58
Restaurant Management Opportunities 0.58
Hiring Restaurant Positions - Servers - Bartender 0.58
Taste Bar Café - Food Sales Associate, Full Time/Part Time: Seattle, WA, Macy’s Downtown Seattle 0.57
Cooks, Servers, Bartenders, Cashiers & Greeters 0.57
Line Cooks, Servers, Cashiers, Bar & Greeters 0.56
Hiring All Restaurant Positions -  Servers - Cooks - Bartenders 0.56
Hiring All Restaurant Positions - Servers -  Cooks - Bartenders 0.56
Hiring All Restaurant Positions - Servers - Cooks - Bartenders 0.56
Hiring All Restaurant Positions - Servers -Cooks - Bartenders 0.56
SERVER / RE STAURANT / HOSPITALITY EXPERIENCE - CUSTOMER RELA TIONS REPS 0.56
Hiring All Restaurant Positions - Servers - Host Staff - Bartenders 0.56
Restaurant Server - Mystic Steakhouse 0.56
Hiring All Restaurant Positions - Cooks - Servers - Bartenders 0.56
Hiring Restaurant Positions - Servers - Barbacks - Host Staff 0.55
Dining Room Server - Restaurant - Los Angeles 0.55
Bartender - Kitchen Service Bar (PT) 0.55
""".split("\n")

# Analyst -> Financial, Marketing and other domain
lst2 = """
~ Click Here ~ Senior Financial Analysts 1.0
Senior Financial Analyst 0.84
Financial Analysts 0.82
Senior Financial Business Analyst 0.77
Financial Analysts needed 0.76
Financial Analysts Needed! 0.76
Junior Financial Analyst 0.76
Calling All Experienced Financial Analysts!! 0.76
Financial  Analyst 0.76
Financial Analyst 0.76
On the Lookout for Accomplished Financial Analysts! 0.75
In need of Financial Analysts 0.75
Financial Analysts Wanted! 0.73
Financial Reporting Analyst 0.73
Financial Analysts - All Levels Needed! 0.73
Searching for All Accurate and Focused Financial Analysts!! 0.72
Searching for Proven Financial Analysts! 0.72
Senior Financial Analyst ~ Apply Now ~ 0.72
Financial Analyst Needed! 0.71
Excellent Opportunities for Financial Analysts 0.71
Senior Business Analyst 0.7
On the Lookout for Dynamic Financial Analysts! 0.7
Proficient Financial Analysts Needed ASAP! 0.7
Financial Analysts Apply Within 0.69
Jr. Financial Analyst 0.69
Success Driven Financial Analysts Needed Now! 0.69
Sr. Financial Analyst Needed! 0.68
Financial/Data Analyst 0.68
Financial Analyst/Pricing Analyst Needed ASAP! 0.67
Sr. Financial Analyst 0.67
Financial Analyst Intern 0.66
Seeking All Skilled and Motived Financial Analysts! 0.66
Entry Level Financial Analyst 0.66
Financial Analyst - Healthcare 0.66
Financial Sales Analyst 0.66
Entry-Level Financial Analyst 0.66
Now Seeking a Financial Analyst! 0.66
Financial Analyst- Wyomissing 0.66
Motivated Financial Analysts Always Wanted!! 0.66
Financial Analyst at Healthcare Organization 0.65
Senior Sales Analyst 0.65
Investment Banking Analyst 0.65
Financial Analyst for a Prestigious Financial Service Firm! 0.64
Financial Analyst- Myerstown 0.64
Financial Analyst  (Advanced Excel) 0.64
Financial Aid Analyst 0.64
Hedge Fund Analyst 0.64
Junior Financial Analyst- Growing Company 0.63
FINANCE BUSINESS ANALYST 0.63
Senior Financial Reporting Manager 0.63
Financial Analyst â Accounting, PT 0.63
Financial Analyst needed for Financial Service firm! 0.63
Financial Analyist 0.63
Financial Analyst - Exciting Opportunity 0.63
Financial Business Analyst II 0.63
Real Estate Investment Firm seeks Financial Analyst! 0.63
Financial Analyst needed for well-known, established Firm! 0.63
Entry Level Corporate Finance Analyst 0.63
Financial Analyst with Advanced Excel 0.62
Senior Accountant - with Financial Edge 0.62
Financial Analyst with exposure to upper management! 0.62
Financial Analyst Needed for Fantastic Opportunity 0.62
Financial Advisor 0.62
Financial Analyst Needed for a fortune 500 company! 0.62
Immediate need for a Financial Analyst 0.62
Analyst Opportunity within the Financial Industy! 0.62
Financial Analyst - Exciting Growing Firm 0.62
Entry Level Financial Data Analyst 0.61
Financial Analyst - Entry Level 0.61
Financial Analyst and Underwriter 0.61
Investment Firm seeks Entry Level Financial Analyst 0.61
Analyst (Data Analyst for Fortune 500 Company!) 0.61
Senior Accountants 0.61
Mortgage Analyst 0.61
Financial Analyst- Morgantown 0.61
Accounting Analyst 0.61
Financial Analyst - HUGE company! 0.6
Senior Accountant - 0.6
SENIOR ACCOUNTANT 0.6
Senior Accountant 0.6
Business Analyst 0.6
Senior Analyst/Accountant at Major Healthcare System 0.6
Junior Financial Analyst Needed for Property Management! 0.6
Financial Aid Analyst/Specialist 0.6
Banking Analyst 0.6
Investor Accountant 0.6
Recent Grads-Accounting & Finance 0.6
Entry Level Investment Analyst 0.59
Data Analyst / Financial Analyst 0.59
Financial Accountant 0.59
IT Finance Reporting & Analystics Intern 0.59
Accounting and Finance Professionals 0.59
Financial Analyst - Automotive (15-00260) 0.59
Financial Specialist 0.59
Financial Reporting Intern 0.59
Financial Asset Liability Management Analyst 0.59
Jr. Business Analyst 0.58
Financial/System/Budget Analyst 0.58
Financial Analyst in North Columbus 0.58
Financial Analyst - South Fort Worth 0.58
Financial Analyst for Software company in Mountain View! 0.58
Financial Analyst/Payroll Associate 0.58
Budget Analyst 0.58
Market Research Analyst 0.58
Mutual Fund Valuation Analyst 0.58
Senior Accountant/Analyst at Prestigious Hospital 0.58
Pricing Analyst 0.57
Intern, Pricing Analyst 0.57
Junior Billing Analyst 0.57
Senior Cost Accountant Wanted - Click here to learn more! 0.57
Mutual Fund Accountant 0.57
Hedge Fund Accountant 0.57
Residential Mortgage Analyst 0.57
Motivated Financial Analyst Needed! 0.57
Sr. Business Analyst 0.57
Entry Level Financial Analyst needed for LA Investment Firm! 0.57
Entry Level Financial Analyst-Strong Excel Needed-Project! 0.57
Mortgage Loan Analyst 0.57
Financial Administrator 0.56
Financial Analyst - Entry Level Opportunity 0.56
College Co-op-Bank Business Analyst 0.56
Financial Analyst Needed in Sandusky 0.56
Financial Analyst in Blue Ash 0.56
Senior Workforce Management and Reporting Analyst 0.56
Executive Assistant - Finance Professional 0.56
Investment Operations Analyst Intern 0.56
Financial Advisor - Investment Advisor 0.56
Financial Planner 0.56
Financial Analyst Needed on the East Side of Indy! 0.56
Entry Level Financial Analyst at exciting DC non-profit 0.56
Financial Reporting Manager 0.56
Intern - Reporting Analyst 0.56
South OKC Company seeking Financial Analyst! 0.56
Administrative Analyst 0.56
Entry Level Investment Operations Analyst 0.56
Marketing Analyst 0.56
Fund Accountant 0.56
Bank Credit Analyst 0.56
Research Analyst 0.55
Purchasing Analyst 0.55
Senior Accountant for High Tech Company 0.55
Highly successful entertainment firm seeks Financial Analyst 0.55
Entry Level Opportunity for Financial Analyst 0.55
Treasury Analyst 0.55
Entry Level Financial Analyst for Non-Profit Organization 0.55
""".split("\n")

## Outliers: CSR from Retail

# Customer Service Or Retail Experience 0.62
# Customer Service- Gain Experience Here! 0.61
# ENTRY LEVEL MARKETING MAVERICKS ★ Sales & Marketing ★ 0.6
# Customer Service/Concierge 0.6
# Retail & Customer Service Experience Wanted for Entry Level 0.59
# Customer Service / Retail (Entry Level) 0.58
# Entry Level Customer Service Opportunity! 0.57
# Experienced Customer Service Individual! 0.57
# ENTRY LEVEL MARKETING/CUSTOMER SERVICE- EVENT PROMOTIONS & RETAIL 0.57
# CUSTOMER SERVICE / RETAIL / SALES 0.56
# Customer Service / Sales ( New Grads Welcome! ) 0.56
# Customer Service Representatives - Part Time Openings! 0.56
# Customer Service Representative Openings! 0.56
# Retail & Customer Service Experience Wanted for Entry Level Management 0.56
# Customer Service/Sales 0.56
# Customer Service / Sales 0.56
# Customer Service - Event / Retail Marketing & Advertising Firm 0.55
# CUSTOMER SERVICE REPRESENTATIVE - Earn & Learn! 0.55
# Customer Experience Specialist 0.55
# .Marketing, Customer Service, Entry Level 0.55

In [ ]:
lst = lst1 + lst2 #+lst3 etc
lst = [el.rsplit(" ", 1)[0].strip() for el in lst if el]

In [ ]:
# Get new examples of job titles (not in verified list)
job_titles_2 = sorted([sk for sk in sorted(job_titles, reverse=True) if sk not in set(lst)], reverse=True)

print(len(job_titles_2))
print(job_titles_2[:10])

curr_skills = np.array(job_titles_2)

curr_embed = model.encode(
        curr_skills,
        batch_size=64,
        show_progress_bar=True
)

In [ ]:
matrix = cos_sim(curr_embed, curr_embed)
len(matrix)

In [ ]:
i = 1
row = matrix[i]

for el in np.argsort(-row):
    if matrix[i][el] < 0.55: break
    print(curr_skills[el], round(matrix[i][el].item(), 2))

#### We can check it manually step by step and formed groups of job title. These Groups can be used as markers for further work.